# Landslide susceptibility in the Koshi Basin: comparative, spatially validated ML study
**Author:** Kaushal Khanal (11495088) · STW7072CEM Machine Learning

Aim: to identify the machine learning model that best predicts landslide susceptibility in the Koshi Basin under
spatially independent validation. Sections 6–7 compare the models, Section 8 compares random and block validation,
Section 9 analyses the controlling factors and Section 10 builds and validates the map.

The **core pipeline (the unmarked cells of Sections 1–11) is unchanged** from the run reported in
the first draft, so its numbers are reproduced exactly with seed 42. Sections marked **[NEW]** add the diagnostics,
statistical tests and validation checks used in the revised paper. Section 12 writes `outputs/paper_values.tex`,
which the LaTeX paper reads, so no number is typed by hand.

## 0. Setup and configuration

In [ ]:
# Colab has pandas, scikit-learn, shapely and seaborn; rasterio (for the DEM) may need installing
import importlib.util, subprocess, sys
for pkg in ['rasterio', 'openpyxl', 'ipywidgets']:
    if importlib.util.find_spec(pkg) is None:
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg])
        print(f'{pkg}:', 'installed' if r.returncode == 0 else 'could not install - run  pip install ' + pkg)
print('Libraries ready (restart the kernel if anything was just installed)')

In [ ]:
import os
os.environ['PYTHONWARNINGS'] = 'ignore'   # inherited by joblib workers, silences the sklearn.utils.parallel warning flood
import warnings, json, time, math, glob, datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# ---------------- CONFIGURATION (edit here only) ----------------
DATA_PATH        = 'Koshi_Landslide_Inventory_2024.xlsx'
DEM_PATH         = 'dem_cop30.tif'     # created in Step 2, or upload your own GeoTIFF (EPSG:4326)
LEGACY_DEM_PATHS = ['dem_srtm.tif']    # the first run saved its Copernicus mosaic under this (misleading) name
OPENTOPO_KEY     = ''                  # optional: OpenTopography key for SRTM; if empty, Copernicus GLO-30 is used
DEM_TYPE         = 'SRTMGL1'
BOUNDARY_PATH    = ''                  # optional basin boundary; '' = concave hull of landslides + 2 km
SEED             = 42
CRS_UTM          = 'EPSG:32645'        # WGS 84 / UTM 45N, same as the inventory
EXCLUDE_BUFFER_M = 50                  # no background point within 50 m of a landslide edge (Gurung et al. 2026)
HULL_BUFFER_M    = 2000
MAX_ELEV_M       = 4500                # inventory mapping limit
BLOCK_SIZE_M     = 10_000              # 10 km spatial blocks
TEST_FRACTION    = 0.20
CV_FOLDS         = 5
SVM_TUNE_MAX     = 6000
GRID_SPACING_M   = 1000
QUICK            = False               # True = tiny grids for a fast test run
OUT              = 'outputs'           # tables, results.json, paper_values.tex
FIG              = 'figures'           # every saved image (same folder name the paper uses)
# [NEW] settings for the added analyses
S2_OVERPASS_UTC  = '2024-12-27T04:52:21'   # sensing time in the Image_ID of 95.5 % of records
SHADOW_MAX_DIST_M = 6000                   # horizon search distance for cast shadows
BLOCK_SIZES_KM   = [1, 2, 5, 10, 20, 40]   # block-size sensitivity
N_REPEATS        = 3 if QUICK else 10      # repeated random-vs-spatial splits
N_BOOT           = 200 if QUICK else 1000  # block-bootstrap replicates

os.makedirs(OUT, exist_ok=True); os.makedirs(FIG, exist_ok=True)
np.random.seed(SEED)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 9})
RESULTS, PAPER = {}, {}

def savefig(name):
    plt.savefig(f'{FIG}/{name}.png', dpi=200, bbox_inches='tight')
    plt.show()

def pv(key, value):
    # Record a value for the paper (written to outputs/paper_values.tex in Section 12).
    PAPER[key] = value
    return value

f3 = lambda v: f'{v:.3f}'

def show_table(df, name, title=None, index=True, fmt='{:.3f}'):
    # Render a results table as a figure, save it as figures/tab_<name>.png and show it.
    d = df.to_frame() if isinstance(df, pd.Series) else df.copy()
    if index: d = d.reset_index()
    def cell(v):
        if isinstance(v, (bool, np.bool_)): return str(v)
        if isinstance(v, (float, np.floating)): return fmt.format(v) if np.isfinite(v) else '–'
        if isinstance(v, (int, np.integer)): return f'{v:,}'
        return str(v)
    rows = [[cell(v) for v in r] for r in d.values]
    nr, nc = len(rows), len(d.columns)
    fig, ax = plt.subplots(figsize=(min(1.45 * nc + 1, 16), 0.3 * (nr + 1) + 0.7)); ax.axis('off')
    t = ax.table(cellText=rows, colLabels=[str(c) for c in d.columns], loc='center', cellLoc='center')
    t.auto_set_font_size(False); t.set_fontsize(8); t.auto_set_column_width(list(range(nc))); t.scale(1, 1.35)
    for (r, c), ce in t.get_celld().items():
        ce.set_edgecolor('#cccccc')
        if r == 0: ce.set_facecolor('#4c72b0'); ce.set_text_props(color='w', weight='bold')
        elif r % 2 == 0: ce.set_facecolor('#f3f3f3')
    if title: ax.set_title(title, fontsize=10, weight='bold', pad=6)
    savefig(f'tab_{name}')

## 1. Load, check and visualise the landslide inventory
Checks: duplicate IDs, missing values/coordinates, area consistency, centroid-vs-Lat/Long offset, implausible slope,
elevation and aspect. Problems are flagged and handled in memory; the source file is never modified.

In [ ]:
if not os.path.exists(DATA_PATH) and os.path.exists(os.path.join('data', DATA_PATH)):
    DATA_PATH = os.path.join('data', DATA_PATH)
if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        print('Upload Koshi_Landslide_Inventory_2024.xlsx:'); files.upload()
    except ImportError:
        raise FileNotFoundError(f'Put {DATA_PATH} in the same folder as this notebook')
ls = pd.read_excel(DATA_PATH, sheet_name='Landslide_Inventory')
print(ls.shape[0], 'landslides,', ls.shape[1], 'columns')
ls.head()

In [ ]:
from pyproj import Transformer

to_utm = Transformer.from_crs('EPSG:4326', CRS_UTM, always_xy=True)
lx, ly = to_utm.transform(ls['Long'].values, ls['Lat'].values)
centroid_offset_m = np.hypot(lx - ls['Centroid_X'], ly - ls['Centroid_Y'])

checks = {
    'records': len(ls),
    'duplicate_ids': int(ls['ID'].duplicated().sum()),
    'rows_with_missing_values': int(ls.isna().any(axis=1).sum()),
    'missing_coordinates': int(ls[['Centroid_X', 'Centroid_Y']].isna().any(axis=1).sum()),
    'non_positive_area': int((ls['Area_'] <= 0).sum()),
    'areakm2_vs_area_m2_mismatch': int((abs(ls['AreaKM2'] * 1e6 - ls['Area_']) > 10).sum()),
    'centroid_over_1km_from_lat_long': int((centroid_offset_m > 1000).sum()),
    'slope_below_5deg': int((ls['Slope'] < 5).sum()),
    'elevation_above_4500m': int((ls['Elevation'] > MAX_ELEV_M).sum()),
    'aspect_out_of_range': int(((ls['Aspect'] < 0) | (ls['Aspect'] > 360)).sum()),
}
RESULTS['data_checks'] = checks
pv('centroidoffsetmedian', f'{np.median(centroid_offset_m):.0f}')
chk = pd.Series(checks).drop('records')
plt.figure(figsize=(7, 3.4))
plt.barh(chk.index, chk.values, color=['#d62728' if v > 0 else '#2ca02c' for v in chk.values])
for i, v in enumerate(chk.values): plt.text(v + 2, i, str(v), va='center')
plt.xlabel('Number of records'); savefig('fig1a_data_checks')
show_table(chk.to_frame('count'), 'data_checks', 'Data-quality checks')

In [ ]:
ls = ls[~ls['ID'].duplicated()].dropna(subset=['Centroid_X', 'Centroid_Y']).copy()
ls['x'], ls['y'] = ls['Centroid_X'], ls['Centroid_Y']
ls['radius_m'] = np.sqrt(ls['Area_'] / np.pi)                 # equal-area circle radius
ls['image_date'] = ls['Image_ID'].str.extract(r'_(\d{8})T')[0]
print(ls['image_date'].value_counts())

# [NEW] geometry facts used in the paper
from scipy.spatial import cKDTree
d_nn, _ = cKDTree(ls[['x', 'y']].values).query(ls[['x', 'y']].values, k=2)
pv('radiusmedian', f"{ls['radius_m'].median():.1f}")
pv('exclusionmedian', f"{ls['radius_m'].median() + EXCLUDE_BUFFER_M:.0f}")
pv('lsspacingmedian', f'{np.median(d_nn[:, 1]):.0f}')
pv('lsspacingpctunder500', f'{100 * np.mean(d_nn[:, 1] < 500):.1f}')
pv('dominantdatepct', f"{100 * ls['image_date'].value_counts(normalize=True).iloc[0]:.1f}")
print({k: PAPER[k] for k in ['radiusmedian', 'exclusionmedian', 'lsspacingmedian', 'lsspacingpctunder500']})

In [ ]:
fig = plt.figure(figsize=(12, 8))
ax1 = fig.add_subplot(2, 2, 1)
for d, c in zip(sorted(ls['image_date'].dropna().unique()), ['#1f77b4', '#ff7f0e', '#2ca02c', '#9467bd']):
    s = ls[ls['image_date'] == d]; ax1.scatter(s.x / 1000, s.y / 1000, s=1.5, c=c, label=f'{d} ({len(s):,})')
ax1.set_aspect('equal'); ax1.legend(markerscale=6, fontsize=7); ax1.set_xlabel('Easting (km)'); ax1.set_ylabel('Northing (km)')
ax2 = fig.add_subplot(2, 2, 2); ax2.hist(np.log10(ls['Area_']), bins=40, color='#8172b2')
ax2.set_xlabel('log10 area (m²)'); ax2.set_ylabel('Landslides')
ax3 = fig.add_subplot(2, 2, 3, projection='polar'); ax3.set_theta_zero_location('N'); ax3.set_theta_direction(-1)
h, e = np.histogram(ls['Aspect'], bins=np.arange(0, 361, 22.5))
ax3.bar(np.radians(e[:-1] + 11.25), h, width=np.radians(22.5), color='#55a868', edgecolor='w')
ax4 = fig.add_subplot(2, 2, 4)
hb = ax4.hexbin(ls['Elevation'], ls['Slope'], gridsize=40, cmap='viridis', mincnt=1)
fig.colorbar(hb, ax=ax4, label='Landslides'); ax4.axvline(MAX_ELEV_M, color='r', ls='--', lw=1)
ax4.set_xlabel('Elevation (m)'); ax4.set_ylabel('Slope (°)')
plt.tight_layout(); savefig('fig1b_inventory_overview')

## 2. DEM and terrain features
Both classes take every feature from **the same DEM**, so the classifier cannot exploit a difference in data source.
Without an OpenTopography key the notebook downloads **Copernicus GLO-30** (a TanDEM-X-derived *surface* model);
that is what the reported run used, so the paper reports Copernicus GLO-30, not SRTM.

In [ ]:
import urllib.request, urllib.error
import rasterio
from rasterio.merge import merge

to_ll = Transformer.from_crs(CRS_UTM, 'EPSG:4326', always_xy=True)
lon, lat = to_ll.transform(ls['x'].values, ls['y'].values)
W, E, S, N = lon.min() - 0.05, lon.max() + 0.05, lat.min() - 0.05, lat.max() + 0.05
print(f'DEM extent: {W:.2f}-{E:.2f} E, {S:.2f}-{N:.2f} N')

DEM_SOURCE = None
if not os.path.exists(DEM_PATH):
    for p in LEGACY_DEM_PATHS:
        if os.path.exists(p):
            DEM_PATH = p
            DEM_SOURCE = 'Copernicus GLO-30 (mosaic from the first run, saved as dem_srtm.tif)'
if os.path.exists(DEM_PATH + '.source.txt'):
    DEM_SOURCE = open(DEM_PATH + '.source.txt').read().strip()

if not os.path.exists(DEM_PATH) and OPENTOPO_KEY.strip():
    url = (f'https://portal.opentopography.org/API/globaldem?demtype={DEM_TYPE}&south={S:.4f}&north={N:.4f}'
           f'&west={W:.4f}&east={E:.4f}&outputFormat=GTiff&API_Key={OPENTOPO_KEY.strip()}')
    print('Downloading SRTM from OpenTopography...'); urllib.request.urlretrieve(url, DEM_PATH)
    DEM_SOURCE = f'NASA SRTM {DEM_TYPE} (OpenTopography)'
elif not os.path.exists(DEM_PATH):
    print('No API key: downloading Copernicus GLO-30 tiles from AWS...')
    os.makedirs('dem_tiles', exist_ok=True); paths = []
    for la in range(math.floor(S), math.floor(N) + 1):
        for lo in range(math.floor(W), math.floor(E) + 1):
            name = f'Copernicus_DSM_COG_10_N{la:02d}_00_E{lo:03d}_00_DEM'; p = f'dem_tiles/{name}.tif'
            if not os.path.exists(p):
                try:
                    urllib.request.urlretrieve(f'https://copernicus-dem-30m.s3.amazonaws.com/{name}/{name}.tif', p)
                except urllib.error.HTTPError:
                    print('  no tile', name); continue
            paths.append(p)
    srcs = [rasterio.open(p) for p in paths]
    mosaic, mosaic_T = merge(srcs, bounds=(W, S, E, N))
    meta = srcs[0].meta.copy(); meta.update(height=mosaic.shape[1], width=mosaic.shape[2], transform=mosaic_T, compress='deflate')
    with rasterio.open(DEM_PATH, 'w', **meta) as dst: dst.write(mosaic)
    for s_ in srcs: s_.close()
    DEM_SOURCE = 'Copernicus GLO-30 (AWS open data)'
if DEM_SOURCE and not os.path.exists(DEM_PATH + '.source.txt'):
    open(DEM_PATH + '.source.txt', 'w').write(DEM_SOURCE)

with rasterio.open(DEM_PATH) as src:
    dem = src.read(1).astype('float32'); T = src.transform; dem_crs = src.crs
    if src.nodata is not None: dem[dem == src.nodata] = np.nan
dem[dem < -500] = np.nan
assert dem_crs.to_epsg() == 4326, 'This notebook expects a DEM in geographic coordinates (EPSG:4326)'
RESULTS['dem_source'] = DEM_SOURCE or 'unknown (set DEM_SOURCE manually)'
print('DEM', dem.shape, '| resolution', round(T.a * 3600, 2), 'arc-seconds | source:', RESULTS['dem_source'])

In [ ]:
step = max(1, dem.shape[1] // 1500); d = dem[::step, ::step]
gy, gx = np.gradient(np.nan_to_num(d, nan=np.nanmean(d)))
az, alt = np.radians(315), np.radians(45)
slope_r = np.arctan(np.hypot(gx, gy) / 30); aspect_r = np.arctan2(-gx, gy)
hill = np.sin(alt) * np.cos(slope_r) + np.cos(alt) * np.sin(slope_r) * np.cos(az - aspect_r)
extent = [T.c, T.c + T.a * dem.shape[1], T.f + T.e * dem.shape[0], T.f]
fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(hill, cmap='gray', extent=extent); im = ax.imshow(d, cmap='terrain', extent=extent, alpha=0.55, vmin=0, vmax=6000)
ax.scatter(lon, lat, s=0.6, c='red', label='Mapped landslides')
fig.colorbar(im, ax=ax, label='Elevation (m)', shrink=0.8)
ax.set_xlabel('Longitude (°E)'); ax.set_ylabel('Latitude (°N)'); ax.legend(markerscale=10, loc='lower left')
savefig('fig2a_dem_hillshade')

In [ ]:
def terrain(lons, lats):
    # Elevation, slope, aspect, TRI and relief at many points, read straight from the DEM array.
    lons, lats = np.asarray(lons), np.asarray(lats)
    cols = np.floor((lons - T.c) / T.a).astype(int); rows = np.floor((lats - T.f) / T.e).astype(int)
    H, Wd = dem.shape
    ok = (rows >= 5) & (rows < H - 5) & (cols >= 5) & (cols < Wd - 5)
    r, c = np.where(ok, rows, 5), np.where(ok, cols, 5)
    z = lambda dr, dc: dem[r + dr, c + dc]
    a, b, cc = z(-1, -1), z(-1, 0), z(-1, 1)
    d, e, f = z(0, -1), z(0, 0), z(0, 1)
    g, h, i = z(1, -1), z(1, 0), z(1, 1)
    dx = abs(T.a) * 111320 * np.cos(np.radians(lats)); dy = abs(T.e) * 110574
    dzdx = ((cc + 2 * f + i) - (a + 2 * d + g)) / (8 * dx)        # Horn (1981)
    dzdy = ((a + 2 * b + cc) - (g + 2 * h + i)) / (8 * dy)
    slope = np.degrees(np.arctan(np.hypot(dzdx, dzdy)))
    aspect = (np.degrees(np.arctan2(-dzdx, -dzdy)) + 360) % 360  # downslope direction, clockwise from north
    tri = np.sqrt(sum((n - e) ** 2 for n in (a, b, cc, d, f, g, h, i)))   # Riley et al. (1999)
    win = np.stack([z(dr, dc) for dr in range(-5, 6) for dc in range(-5, 6)])
    relief = np.nanmax(win, 0) - np.nanmin(win, 0)                # 11 x 11 window, about 330 m
    out = pd.DataFrame({'elevation': e, 'slope': slope, 'aspect': aspect, 'tri': tri, 'relief': relief})
    out.loc[~ok] = np.nan
    return out

# Sanity check on a synthetic plane rising to the east: aspect must be 270 deg (faces west)
_dem, _T = dem, T
dem = np.tile(np.arange(50, dtype='float32') * 10, (50, 1)); T = rasterio.Affine(1/3600, 0, 86, 0, -1/3600, 27)
_t = terrain([86 + 25/3600], [27 - 25/3600])
print('check: slope %.1f deg, aspect %.0f deg' % (_t.slope[0], _t.aspect[0])); assert abs(_t.aspect[0] - 270) < 1
dem, T = _dem, _T

### 2b. [NEW] Acquisition geometry: solar illumination and cast shadow
95.5 % of the inventory was digitised from one Sentinel-2 scene sensed at 04:52 UTC on 27 Dec 2024. We compute the
sun's position at that moment (NOAA algorithm), the local illumination cos *i* of every sample, and whether the
sample is in a terrain cast shadow (horizon angle towards the sun > sun elevation). These are **not model features**;
they are used in Section 9 to test whether the aspect signal is partly a mapping (visibility) artefact.

In [ ]:
def sun_position(when_utc, lat, lon):
    # NOAA solar position. Returns (elevation, azimuth clockwise from north) in degrees.
    jd = when_utc.timestamp() / 86400 + 2440587.5; Tc = (jd - 2451545) / 36525
    L0 = (280.46646 + Tc * (36000.76983 + 0.0003032 * Tc)) % 360
    M = np.radians(357.52911 + Tc * (35999.05029 - 0.0001537 * Tc)); ecc = 0.016708634 - Tc * (0.000042037 + 1.267e-7 * Tc)
    C = np.sin(M) * (1.914602 - Tc * (0.004817 + 1.4e-5 * Tc)) + np.sin(2 * M) * (0.019993 - 0.000101 * Tc) + np.sin(3 * M) * 0.000289
    om = np.radians(125.04 - 1934.136 * Tc); lam = np.radians(L0 + C - 0.00569 - 0.00478 * np.sin(om))
    eps = np.radians(23 + (26 + (21.448 - Tc * (46.815 + Tc * (0.00059 - Tc * 0.001813))) / 60) / 60 + 0.00256 * np.cos(om))
    decl = np.arcsin(np.sin(eps) * np.sin(lam)); yv = np.tan(eps / 2) ** 2; L0r = np.radians(L0)
    eqt = 4 * np.degrees(yv * np.sin(2 * L0r) - 2 * ecc * np.sin(M) + 4 * ecc * yv * np.sin(M) * np.cos(2 * L0r)
                         - 0.5 * yv * yv * np.sin(4 * L0r) - 1.25 * ecc * ecc * np.sin(2 * M))
    tst = (when_utc.hour * 60 + when_utc.minute + when_utc.second / 60 + eqt + 4 * lon) % 1440
    ha = np.radians(tst / 4 - 180); la = np.radians(lat)
    zen = np.arccos(np.clip(np.sin(la) * np.sin(decl) + np.cos(la) * np.cos(decl) * np.cos(ha), -1, 1))
    azi = (np.degrees(np.arctan2(np.sin(ha), np.cos(ha) * np.sin(la) - np.tan(decl) * np.cos(la))) + 180) % 360
    return 90 - np.degrees(zen), azi

def illumination(slope_deg, aspect_deg, sun_el, sun_az):
    # cos of the solar incidence angle on a tilted surface (1 = facing the sun, <0 = self-shadowed).
    sz, s, a = np.radians(90 - sun_el), np.radians(slope_deg), np.radians(aspect_deg)
    return np.cos(sz) * np.cos(s) + np.sin(sz) * np.sin(s) * np.cos(np.radians(sun_az) - a)

def dem_at(lons, lats):
    cols = np.floor((np.asarray(lons) - T.c) / T.a).astype(int); rows = np.floor((np.asarray(lats) - T.f) / T.e).astype(int)
    ok = (rows >= 0) & (rows < dem.shape[0]) & (cols >= 0) & (cols < dem.shape[1])
    out = np.full(len(rows), np.nan, dtype='float32'); out[ok] = dem[rows[ok], cols[ok]]
    return out

def cast_shadow(lons, lats, sun_el, sun_az, max_d=SHADOW_MAX_DIST_M, chunk=4000):
    # True where terrain towards the sun rises above the sun's elevation (ray march at ~1 pixel steps).
    lons, lats = np.asarray(lons, float), np.asarray(lats, float)
    stepm = abs(T.e) * 110574; dist = np.arange(stepm, max_d + stepm, stepm)
    res = np.zeros(len(lons), bool); hor = np.zeros(len(lons))
    for k in range(0, len(lons), chunk):
        lo_, la_ = lons[k:k + chunk], lats[k:k + chunk]; z0 = dem_at(lo_, la_)
        dlon = dist[None, :] * np.sin(np.radians(sun_az)) / (111320 * np.cos(np.radians(la_))[:, None])
        dlat = dist[None, :] * np.cos(np.radians(sun_az)) / 110574
        zz = dem_at((lo_[:, None] + dlon).ravel(), (la_[:, None] + dlat).ravel()).reshape(len(lo_), len(dist))
        ang = np.degrees(np.arctan((zz - z0[:, None]) / dist[None, :]))
        h_ = np.nanmax(np.where(np.isnan(ang), -90, ang), 1); hor[k:k + chunk] = h_; res[k:k + chunk] = h_ > sun_el
    return res, hor

when = dt.datetime.fromisoformat(S2_OVERPASS_UTC).replace(tzinfo=dt.timezone.utc)
SUN_EL, SUN_AZ = sun_position(when, float(np.mean(lat)), float(np.mean(lon)))
print(f'Sun at overpass: elevation {SUN_EL:.1f} deg, azimuth {SUN_AZ:.1f} deg')
pv('sunel', f'{SUN_EL:.1f}'); pv('sunaz', f'{SUN_AZ:.1f}')
for nm, asp in [('south', 180), ('north', 0)]:
    pv(f'cosi{nm}30', f'{illumination(30, asp, SUN_EL, SUN_AZ):.2f}')
print('cos i for a 30 deg slope: south', PAPER['cosisouth30'], '| north', PAPER['cosinorth30'])

In [ ]:
ls_t = terrain(lon, lat)
ls_feat = pd.concat([ls[['ID', 'x', 'y']].reset_index(drop=True), ls_t], axis=1)
n0 = len(ls_feat); ls_feat = ls_feat.dropna(); dropped_nodata = n0 - len(ls_feat)
dropped_high = int((ls_feat['elevation'] > MAX_ELEV_M).sum())
ls_feat = ls_feat[ls_feat['elevation'] <= MAX_ELEV_M]; ls_feat['label'] = 1
print(f'Landslides kept: {len(ls_feat)} (dropped {dropped_nodata} DEM no-data, {dropped_high} above {MAX_ELEV_M} m)')
RESULTS['landslides_used'] = len(ls_feat); RESULTS['landslides_dropped_high'] = dropped_high

m = ls[['ID', 'Slope', 'Elevation', 'Aspect']].merge(ls_feat[['ID', 'slope', 'elevation', 'aspect']], on='ID')
RESULTS['inventory_vs_dem_corr'] = {'slope': round(float(m['Slope'].corr(m['slope'])), 3),
                                    'elevation': round(float(m['Elevation'].corr(m['elevation'])), 3)}
# [NEW] how far apart the two slope estimates are, and is the inventory slope systematically lower?
pv('slopebias', f"{(m['slope'] - m['Slope']).median():+.1f}")
pv('slopemad', f"{(m['slope'] - m['Slope']).abs().median():.1f}")
dasp = np.abs(((m['aspect'] - m['Aspect']) + 180) % 360 - 180)
pv('aspectagree45', f'{100 * np.mean(dasp <= 45):.1f}')
print(RESULTS['inventory_vs_dem_corr'], {k: PAPER[k] for k in ['slopebias', 'slopemad', 'aspectagree45']})
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(m['Elevation'], m['elevation'], s=2, alpha=.3); ax[0].plot([0, 5500], [0, 5500], 'r--', lw=1)
ax[0].set_xlabel('Inventory elevation (m)'); ax[0].set_ylabel('DEM elevation (m)')
ax[1].scatter(m['Slope'], m['slope'], s=2, alpha=.3); ax[1].plot([0, 70], [0, 70], 'r--', lw=1)
ax[1].set_xlabel('Inventory slope (°)'); ax[1].set_ylabel('DEM slope at centroid (°)')
plt.tight_layout(); savefig('fig2b_inventory_vs_dem')

## 3. Study area and background (class 0) samples
Background points are drawn uniformly inside the study area, below 4,500 m, on valid DEM cells, and farther than
(equal-area radius + 50 m) from every landslide centroid. Sampling stops at a 1:1 ratio.

In [ ]:
import shapely
from shapely.geometry import Point

if BOUNDARY_PATH:
    import geopandas as gpd
    area = gpd.read_file(BOUNDARY_PATH).to_crs(CRS_UTM).union_all()
else:
    area = shapely.concave_hull(shapely.MultiPoint(ls_feat[['x', 'y']].values), ratio=0.3).buffer(HULL_BUFFER_M)
print(f'Study area: {area.area / 1e6:,.0f} km2'); RESULTS['study_area_km2'] = round(area.area / 1e6)

tree, rad = cKDTree(ls[['x', 'y']].values), ls['radius_m'].values
def too_close(xs, ys, k=20):
    d, j = tree.query(np.c_[xs, ys], k=k)
    return (d < rad[j] + EXCLUDE_BUFFER_M).any(axis=1)

rng = np.random.default_rng(SEED)
minx, miny, maxx, maxy = area.bounds
need, bg, tried = len(ls_feat), [], 0
while sum(len(b) for b in bg) < need:
    cand = pd.DataFrame({'x': rng.uniform(minx, maxx, 40000), 'y': rng.uniform(miny, maxy, 40000)}); tried += len(cand)
    cand = cand[shapely.contains_xy(area, cand['x'].values, cand['y'].values)]
    cand = cand[~too_close(cand['x'].values, cand['y'].values)]
    blon, blat = to_ll.transform(cand['x'].values, cand['y'].values)
    t = terrain(blon, blat); t.index = cand.index
    g = pd.concat([cand, t], axis=1).dropna(); bg.append(g[g['elevation'] <= MAX_ELEV_M])
bg = pd.concat(bg).iloc[:need].reset_index(drop=True); bg['ID'] = -1; bg['label'] = 0

data = pd.concat([ls_feat, bg[ls_feat.columns]], ignore_index=True)
data['aspect_sin'] = np.sin(np.radians(data['aspect'])); data['aspect_cos'] = np.cos(np.radians(data['aspect']))
data['x_km'], data['y_km'] = data['x'] / 1000, data['y'] / 1000
TERRAIN = ['elevation', 'slope', 'aspect_sin', 'aspect_cos', 'tri', 'relief']
FEATURES = TERRAIN + ['x_km', 'y_km']
RESULTS['n_samples'] = len(data)
print(f'Candidate points generated: {tried:,}; accepted: {len(bg):,}')

# [NEW] acquisition-geometry attributes (diagnostics only, never model inputs)
dlon_, dlat_ = to_ll.transform(data['x'].values, data['y'].values)
data['cos_i'] = illumination(data['slope'], data['aspect'], SUN_EL, SUN_AZ)
data['shadow'], data['horizon'] = cast_shadow(dlon_, dlat_, SUN_EL, SUN_AZ)
data.to_csv(f'{OUT}/model_dataset.csv', index=False)
print(data['label'].value_counts().rename({1: 'landslide', 0: 'background'}))

In [ ]:
from matplotlib.patches import Circle
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={'width_ratios': [2.2, 1]})
for poly in shapely.get_parts(area):
    hx, hy = poly.exterior.xy; ax[0].plot(hx, hy, 'k-', lw=1)
ax[0].scatter(bg.x, bg.y, s=1, c='#4c72b0', alpha=.5, label='Background (0)')
ax[0].scatter(ls_feat.x, ls_feat.y, s=1, c='#c44e52', alpha=.5, label='Landslide (1)')
ax[0].set_aspect('equal'); ax[0].legend(markerscale=8); ax[0].set_xlabel('Easting (m)'); ax[0].set_ylabel('Northing (m)')
Hd, xe, ye = np.histogram2d(ls.x, ls.y, bins=60); i, j = np.unravel_index(Hd.argmax(), Hd.shape)
cx, cy, r = (xe[i] + xe[i + 1]) / 2, (ye[j] + ye[j + 1]) / 2, 2000
near = ls[ls.x.between(cx - r, cx + r) & ls.y.between(cy - r, cy + r)]
for _, s in near.iterrows():
    ax[1].add_patch(Circle((s.x, s.y), s.radius_m + EXCLUDE_BUFFER_M, fc='#f4a6a6', ec='none'))
    ax[1].add_patch(Circle((s.x, s.y), s.radius_m, fc='#c44e52', ec='none'))
z = bg[bg.x.between(cx - r, cx + r) & bg.y.between(cy - r, cy + r)]
ax[1].scatter(z.x, z.y, s=12, c='#4c72b0'); ax[1].set_xlim(cx - r, cx + r); ax[1].set_ylim(cy - r, cy + r)
ax[1].set_aspect('equal'); ax[1].set_xticks([]); ax[1].set_yticks([])
plt.tight_layout(); savefig('fig3a_study_area_samples')

## 4. Exploratory analysis, collinearity, PCA and K-means

In [ ]:
import seaborn as sns
fig, axs = plt.subplots(2, 3, figsize=(12, 6.5))
for ax, f in zip(axs.ravel(), ['elevation', 'slope', 'aspect', 'tri', 'relief', 'y_km']):
    sns.kdeplot(data=data, x=f, hue='label', common_norm=False, fill=True, ax=ax, palette=['#4c72b0', '#c44e52'])
plt.tight_layout(); savefig('fig3_class_distributions')

fig, axs = plt.subplots(1, 5, figsize=(14, 3.6))
for ax, f in zip(axs, ['elevation', 'slope', 'tri', 'relief', 'x_km']):
    sns.boxplot(data=data, x='label', y=f, ax=ax, palette=['#4c72b0', '#c44e52'], showfliers=False)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['Background', 'Landslide']); ax.set_xlabel('')
plt.tight_layout(); savefig('fig3b_boxplots')

fig = plt.figure(figsize=(8, 4))
for k, (lab, col) in enumerate([(0, '#4c72b0'), (1, '#c44e52')]):
    ax = fig.add_subplot(1, 2, k + 1, projection='polar'); ax.set_theta_zero_location('N'); ax.set_theta_direction(-1)
    h, e = np.histogram(data.loc[data.label == lab, 'aspect'], bins=np.arange(0, 361, 22.5))
    ax.bar(np.radians(e[:-1] + 11.25), h / h.sum(), width=np.radians(22.5), color=col, edgecolor='w')
    ax.set_xlabel(['Background aspect', 'Landslide aspect'][lab]); ax.set_yticklabels([])
plt.tight_layout(); savefig('fig3c_aspect_roses')

plt.figure(figsize=(6, 5))
sns.heatmap(data[FEATURES].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
savefig('fig4_correlation')

In [ ]:
# [NEW] Variance inflation factors: how much collinearity the linear / Bayesian models have to absorb
def vif(df):
    Z = (df - df.mean()) / df.std(); out = {}
    for c in Z.columns:
        A = np.c_[np.ones(len(Z)), Z.drop(columns=c).values]
        beta, *_ = np.linalg.lstsq(A, Z[c].values, rcond=None)
        r2 = 1 - np.sum((Z[c].values - A @ beta) ** 2) / np.sum((Z[c].values - Z[c].mean()) ** 2)
        out[c] = 1 / max(1 - r2, 1e-9)
    return pd.Series(out)
V = vif(data[FEATURES]); RESULTS['vif'] = V.round(2).to_dict()
for c in ['slope', 'tri', 'relief', 'elevation', 'y_km']: pv(f'vif{c.replace("_", "")}', f'{V[c]:.1f}')
pv('corrslopetri', f"{data['slope'].corr(data['tri']):.2f}")
show_table(V.to_frame('VIF'), 'vif', 'Variance inflation factors', fmt='{:.2f}')

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
Z = StandardScaler().fit_transform(data[TERRAIN]); pca = PCA().fit(Z)
RESULTS['pca_explained_variance'] = [round(float(v), 3) for v in pca.explained_variance_ratio_]
pv('pcaone', f'{100 * pca.explained_variance_ratio_[0]:.1f}'); pv('pcathree', f'{100 * pca.explained_variance_ratio_[:3].sum():.1f}')
fig, ax = plt.subplots(1, 3, figsize=(15, 4)); k = np.arange(1, len(TERRAIN) + 1)
ax[0].bar(k, pca.explained_variance_ratio_ * 100); ax[0].plot(k, np.cumsum(pca.explained_variance_ratio_) * 100, 'r-o')
ax[0].set_xlabel('Component'); ax[0].set_ylabel('% variance')
P = pca.transform(Z)[:, :2]; s = rng.choice(len(P), min(4000, len(P)), replace=False)
ax[1].scatter(P[s, 0], P[s, 1], c=data['label'].values[s], cmap='coolwarm', s=3, alpha=.6); ax[1].set_xlabel('PC1'); ax[1].set_ylabel('PC2')
load = pd.DataFrame(pca.components_[:3].T, index=TERRAIN, columns=['PC1', 'PC2', 'PC3'])
sns.heatmap(load, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax[2])
plt.tight_layout(); savefig('fig5_pca')
show_table(load, 'pca_loadings', 'PCA loadings (PC1–PC3)', fmt='{:.2f}')

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
scL = StandardScaler().fit(data.loc[data.label == 1, TERRAIN])        # kept: reused in Section 7 [NEW]
ZL = scL.transform(data.loc[data.label == 1, TERRAIN])
ks = range(2, 8); sil, inertia = {}, {}
for k in ks:
    km_k = KMeans(k, n_init=10, random_state=SEED).fit(ZL)
    sil[k] = silhouette_score(ZL, km_k.labels_, sample_size=4000, random_state=SEED); inertia[k] = km_k.inertia_
k_best = max(sil, key=sil.get); km = KMeans(k_best, n_init=10, random_state=SEED).fit(ZL)
RESULTS['kmeans'] = {'silhouette': {k: round(float(v), 3) for k, v in sil.items()}, 'k_best': int(k_best)}
pv('silbest', f'{sil[k_best]:.3f}'); pv('silrange', f'{min(sil.values()):.3f}--{max(sil.values()):.3f}')
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(list(inertia), list(inertia.values()), 'o-'); ax[0].set_xlabel('k'); ax[0].set_ylabel('Inertia')
ax[1].plot(list(sil), list(sil.values()), 'o-', c='#dd8452'); ax[1].axvline(k_best, ls='--', c='k'); ax[1].set_xlabel('k'); ax[1].set_ylabel('Silhouette')
lsx = data.loc[data.label == 1]
ax[2].scatter(lsx.x_km, lsx.y_km, c=km.labels_, cmap='tab10', s=1.5); ax[2].set_aspect('equal')
plt.tight_layout(); savefig('fig5b_kmeans')
prof = lsx[['elevation', 'slope', 'tri', 'relief']].assign(cluster=km.labels_).groupby('cluster').mean()
show_table(prof.assign(n=np.bincount(km.labels_)), 'kmeans_profiles', 'K-means landslide regimes (cluster means)', fmt='{:.1f}')

## 5. Spatial blocks and train/test split
10 km × 10 km blocks; 20 % of blocks held out; tuning by 5-fold GroupKFold on the remaining blocks.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
block_id = lambda xs, ys, size=BLOCK_SIZE_M: (np.floor(np.asarray(xs) / size).astype(int).astype(str) + '_' +
                                           np.floor(np.asarray(ys) / size).astype(int).astype(str))
data['block'] = block_id(data['x'], data['y'])
X, y, groups = data[FEATURES].astype('float64'), data['label'].values, data['block'].values
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=SEED).split(X, y, groups))
print(f'Blocks: {data.block.nunique()} | train {len(tr)} points, test {len(te)} points')
RESULTS['split'] = {'blocks': int(data.block.nunique()), 'train': int(len(tr)), 'test': int(len(te)),
                    'test_blocks': int(len(np.unique(groups[te]))), 'test_landslide_share': round(float(y[te].mean()), 3)}
pv('testblocks', str(RESULTS['split']['test_blocks'])); pv('trainblocks', str(data.block.nunique() - RESULTS['split']['test_blocks']))
pv('ptsperblockmedian', f"{data.groupby('block').size().median():.0f}")
print(RESULTS['split'])
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={'width_ratios': [2.2, 1]})
ax[0].scatter(data.x_km.iloc[tr], data.y_km.iloc[tr], s=1, c='#999999', label='train blocks')
ax[0].scatter(data.x_km.iloc[te], data.y_km.iloc[te], s=1, c='#d62728', label='test blocks')
ax[0].set_aspect('equal'); ax[0].legend(markerscale=8); ax[0].set_xlabel('Easting (km)'); ax[0].set_ylabel('Northing (km)')
data.groupby('block').size().plot.hist(bins=30, ax=ax[1], color='#4c72b0'); ax[1].set_xlabel('Points per block')
plt.tight_layout(); savefig('fig6_spatial_blocks')

## 6. Models, hyperparameter grids and spatial cross-validation
Scaling sits *inside* each pipeline so it is re-fitted on every training fold (no leakage). Tree models are scale-free.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.base import clone

sc = lambda m: Pipeline([('scale', StandardScaler()), ('model', m)])
q = QUICK
MODELS = {
  'Baseline (majority)': (DummyClassifier(strategy='prior'), {}),
  'Logistic Regression': (sc(LogisticRegression(max_iter=3000)), {'model__C': [0.1, 1] if q else [0.01, 0.1, 1, 10]}),
  'LDA':                 (sc(LinearDiscriminantAnalysis()), {}),
  'Gaussian Naive Bayes':(sc(GaussianNB()), {}),
  'KNN':                 (sc(KNeighborsClassifier()), {'model__n_neighbors': [15] if q else [5, 15, 31, 51],
                                                       'model__weights': ['distance'] if q else ['uniform', 'distance']}),
  'SVM (RBF)':           (sc(SVC(kernel='rbf', random_state=SEED)), {'model__C': [1] if q else [1, 10],
                                                                    'model__gamma': ['scale'] if q else ['scale', 0.1]}),
  'Decision Tree':       (DecisionTreeClassifier(random_state=SEED), {'max_depth': [5] if q else [3, 5, 8, 12, None],
                                                                    'min_samples_leaf': [10] if q else [1, 10, 50]}),
  'Random Forest':       (RandomForestClassifier(n_estimators=100 if q else 300, n_jobs=-1, random_state=SEED),
                          {'max_depth': [10] if q else [10, 20, None], 'min_samples_leaf': [5] if q else [1, 5]}),
  'Extra Trees':         (ExtraTreesClassifier(n_estimators=100 if q else 300, n_jobs=-1, random_state=SEED),
                          {'max_depth': [10] if q else [10, 20, None], 'min_samples_leaf': [5] if q else [1, 5]}),
  'Gradient Boosting':   (HistGradientBoostingClassifier(max_iter=100 if q else 300, random_state=SEED),
                          {'learning_rate': [0.1] if q else [0.05, 0.1], 'max_depth': [7] if q else [3, 7, None]}),
}

def score(m, Xs):
    return m.predict_proba(Xs)[:, 1] if hasattr(m, 'predict_proba') else m.decision_function(Xs)

def fit_all(Xtr, ytr, gtr, cv):
    fitted = {}
    for name, (est, grid) in MODELS.items():
        t0 = time.time()
        if name.startswith('SVM') and len(Xtr) > SVM_TUNE_MAX:
            idx = np.random.default_rng(SEED).choice(len(Xtr), SVM_TUNE_MAX, replace=False)
            gs = GridSearchCV(est, grid, cv=cv, scoring='roc_auc', n_jobs=-1)
            gs.fit(Xtr.iloc[idx], ytr[idx], groups=None if gtr is None else gtr[idx]); best = gs.best_estimator_.fit(Xtr, ytr)
        else:
            gs = GridSearchCV(est, grid or {}, cv=cv, scoring='roc_auc', n_jobs=-1)
            gs.fit(Xtr, ytr, groups=gtr); best = gs.best_estimator_
        fitted[name] = dict(model=best, params=gs.best_params_, cv_auc=gs.best_score_,
                            cv_std=gs.cv_results_['std_test_score'][gs.best_index_], secs=time.time() - t0,
                            cv_table=pd.DataFrame(gs.cv_results_))
        print(f"{name:22s} CV AUC {gs.best_score_:.3f} ± {fitted[name]['cv_std']:.3f}  {gs.best_params_}  ({time.time()-t0:.0f}s)")
    return fitted

print('Spatial cross-validation (GroupKFold on blocks):')
spatial = fit_all(X.iloc[tr], y[tr], groups[tr], GroupKFold(n_splits=CV_FOLDS))

In [ ]:
cvdf = pd.DataFrame({n: {'CV AUC': f['cv_auc'], 'std': f['cv_std'], 'secs': f['secs']} for n, f in spatial.items()}).T.sort_values('CV AUC')
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].barh(cvdf.index, cvdf['CV AUC'], xerr=cvdf['std'], color='#4c72b0', capsize=3); ax[0].axvline(0.5, c='k', ls='--', lw=.8)
ax[0].set_xlim(0.4, 1); ax[0].set_xlabel('Spatial CV ROC-AUC (mean ± std)')
ax[1].barh(cvdf.index, cvdf['secs'], color='#dd8452'); ax[1].set_xscale('log'); ax[1].set_xlabel('Tuning + training time (s)')
plt.tight_layout(); savefig('fig6b_cv_auc_and_time')

### 6b. [NEW] Per-algorithm diagnostics
Each algorithm has a specific job in this study (see paper Section 6). These checks measure whether the fitted model
actually behaves as that job requires, using the spatial training/test split only.

In [ ]:
from sklearn.metrics import roc_auc_score
Xtr_, Xte_, ytr_, yte_, gtr_ = X.iloc[tr], X.iloc[te], y[tr], y[te], groups[tr]
DIAG = {}
gkf = GroupKFold(n_splits=CV_FOLDS)

# Baseline: which class does the prior strategy predict, and why
DIAG['baseline_pred_class'] = int(spatial['Baseline (majority)']['model'].predict(Xte_.iloc[:1])[0])
DIAG['train_class_counts'] = np.bincount(ytr_).tolist()

# Logistic Regression: standardised coefficients and odds ratios per +1 SD
lr = spatial['Logistic Regression']['model'].named_steps['model']
coef = pd.Series(lr.coef_[0], index=FEATURES)
DIAG['lr_coef'] = coef.round(3).to_dict(); DIAG['lr_or'] = np.exp(coef).round(3).to_dict()
pv('lrcosOR', f"{np.exp(coef['aspect_cos']):.2f}"); pv('lrcoscoef', f"{coef['aspect_cos']:+.2f}")
pv('lrslopecoef', f"{coef['slope']:+.2f}"); pv('lrtricoef', f"{coef['tri']:+.2f}")

# LDA: equal-covariance assumption, tested by QDA (same features, class-specific covariances)
qda = sc(QuadraticDiscriminantAnalysis(reg_param=0.0)).fit(Xtr_, ytr_)
DIAG['qda_test_auc'] = roc_auc_score(yte_, qda.predict_proba(Xte_)[:, 1])
Zs = StandardScaler().fit(Xtr_).transform(Xtr_)
ld = [np.linalg.slogdet(np.cov(Zs[ytr_ == k].T))[1] for k in (0, 1)]
DIAG['logdet_cov_bg_ls'] = [round(v, 2) for v in ld]
pv('qdaauc', f3(DIAG['qda_test_auc']))

# Gaussian NB: cost of the independence assumption (drop the redundant slope, or decorrelate with PCA first)
def nb_auc(cols=None, whiten=False):
    steps = [('scale', StandardScaler())] + ([('pca', PCA(whiten=True))] if whiten else []) + [('model', GaussianNB())]
    c = cols or FEATURES; m_ = Pipeline(steps).fit(Xtr_[c], ytr_)
    return roc_auc_score(yte_, m_.predict_proba(Xte_[c])[:, 1])
DIAG['nb_no_slope'] = nb_auc([f for f in FEATURES if f != 'slope'])
DIAG['nb_no_slope_relief'] = nb_auc([f for f in FEATURES if f not in ('slope', 'relief')])
DIAG['nb_pca_whitened'] = nb_auc(whiten=True)
pv('nbnoslope', f3(DIAG['nb_no_slope'])); pv('nbnosloperelief', f3(DIAG['nb_no_slope_relief'])); pv('nbpca', f3(DIAG['nb_pca_whitened']))

# KNN: self-match artefact behind the training AUC of 1.000
knn = spatial['KNN']['model']
knn_u = clone(knn).set_params(model__weights='uniform').fit(Xtr_, ytr_)
DIAG['knn_train_auc_uniform'] = roc_auc_score(ytr_, knn_u.predict_proba(Xtr_)[:, 1])
DIAG['knn_test_auc_uniform'] = roc_auc_score(yte_, knn_u.predict_proba(Xte_)[:, 1])
pv('knntrainuniform', f3(DIAG['knn_train_auc_uniform'])); pv('knntestuniform', f3(DIAG['knn_test_auc_uniform']))

# SVM: share of training points that became support vectors (a direct measure of class overlap)
svm = spatial['SVM (RBF)']['model'].named_steps['model']
DIAG['svm_sv_fraction'] = float(svm.n_support_.sum() / len(Xtr_)); DIAG['svm_n_sv'] = svm.n_support_.tolist()
pv('svmsvfrac', f"{100 * DIAG['svm_sv_fraction']:.1f}")

# Decision tree: spatial-CV AUC and training AUC against depth
depths = list(range(1, 21)); dt_cv, dt_tr = [], []
for dpt in depths:
    m_ = DecisionTreeClassifier(max_depth=dpt, min_samples_leaf=50, random_state=SEED)
    aucs = [roc_auc_score(ytr_[v], clone(m_).fit(Xtr_.iloc[t_], ytr_[t_]).predict_proba(Xtr_.iloc[v])[:, 1])
            for t_, v in gkf.split(Xtr_, ytr_, gtr_)]
    dt_cv.append(np.mean(aucs)); dt_tr.append(roc_auc_score(ytr_, m_.fit(Xtr_, ytr_).predict_proba(Xtr_)[:, 1]))
DIAG['dt_depth_peak'] = depths[int(np.argmax(dt_cv))]
pv('dtdepthpeak', str(DIAG['dt_depth_peak'])); pv('dtcvpeak', f3(max(dt_cv)))
dt_full = DecisionTreeClassifier(random_state=SEED).fit(Xtr_, ytr_)
DIAG['dt_unlimited_depth'] = int(dt_full.get_depth()); DIAG['dt_unlimited_leaves'] = int(dt_full.get_n_leaves())
DIAG['dt_unlimited_test_auc'] = roc_auc_score(yte_, dt_full.predict_proba(Xte_)[:, 1])
pv('dtfulldepth', str(DIAG['dt_unlimited_depth'])); pv('dtfullleaves', f"{DIAG['dt_unlimited_leaves']:,}"); pv('dtfullauc', f3(DIAG['dt_unlimited_test_auc']))

# Random Forest / Extra Trees: tree size, and RF out-of-bag AUC (a random-resampling estimate)
for nm in ['Random Forest', 'Extra Trees']:
    est = spatial[nm]['model'].estimators_
    DIAG[f'{nm}_mean_depth'] = float(np.mean([e_.get_depth() for e_ in est]))
    DIAG[f'{nm}_mean_leaves'] = float(np.mean([e_.get_n_leaves() for e_ in est]))
pv('rfdepth', f"{DIAG['Random Forest_mean_depth']:.1f}"); pv('rfleaves', f"{DIAG['Random Forest_mean_leaves']:,.0f}")
pv('etdepth', f"{DIAG['Extra Trees_mean_depth']:.1f}"); pv('etleaves', f"{DIAG['Extra Trees_mean_leaves']:,.0f}")
rf_oob = clone(spatial['Random Forest']['model']).set_params(oob_score=True).fit(Xtr_, ytr_)
DIAG['rf_oob_auc'] = roc_auc_score(ytr_, rf_oob.oob_decision_function_[:, 1]); pv('rfoobauc', f3(DIAG['rf_oob_auc']))

# Gradient Boosting: early stopping is automatic for n > 10,000 and uses a *random* 10 % validation split
gb = spatial['Gradient Boosting']['model']
DIAG['gb_n_iter'] = int(gb.n_iter_); DIAG['gb_early_stopping_active'] = bool(gb.do_early_stopping_)
gb_noes = clone(gb).set_params(early_stopping=False).fit(Xtr_, ytr_)
DIAG['gb_noes_test_auc'] = roc_auc_score(yte_, gb_noes.predict_proba(Xte_)[:, 1])
pv('gbniter', str(DIAG['gb_n_iter'])); pv('gbnoesauc', f3(DIAG['gb_noes_test_auc']))
pv('gbmaxleaf', str(gb.max_leaf_nodes))
RESULTS['diagnostics'] = {k: (round(v, 4) if isinstance(v, float) else v) for k, v in DIAG.items()}
show_table(pd.Series({k: v for k, v in RESULTS['diagnostics'].items() if not isinstance(v, (dict, list))}, dtype=object).to_frame('value'), 'diagnostics', 'Per-algorithm diagnostics')

In [ ]:
# Visual [NEW]: decision-tree depth curve (overfitting onset) — column format for the paper
fig, ax = plt.subplots(figsize=(4.2, 2.8))
ax.plot(depths, dt_tr, 'o-', ms=3, c='#dd8452', label='Training AUC'); ax.plot(depths, dt_cv, 'o-', ms=3, c='#4c72b0', label='Spatial CV AUC')
ax.axvline(DIAG['dt_depth_peak'], c='k', ls='--', lw=.8); ax.set_xlabel('Maximum depth (min. leaf = 50)'); ax.set_ylabel('ROC-AUC')
ax.legend(fontsize=7); plt.tight_layout(); savefig('figH_tree_depth_col')

## 7. Evaluation on spatially held-out test blocks

In [ ]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
                             average_precision_score, matthews_corrcoef, brier_score_loss, confusion_matrix, roc_curve,
                             precision_recall_curve)

def evaluate(fitted, Xtr, ytr, Xte, yte):
    rows = []
    for name, f in fitted.items():
        m = f['model']; p = m.predict(Xte); s = score(m, Xte)
        rows.append({'Model': name, 'Accuracy': accuracy_score(yte, p), 'Balanced acc.': balanced_accuracy_score(yte, p),
                     'Precision': precision_score(yte, p, zero_division=0), 'Recall': recall_score(yte, p),
                     'F1': f1_score(yte, p, zero_division=0), 'ROC-AUC': roc_auc_score(yte, s),
                     'PR-AUC': average_precision_score(yte, s), 'MCC': matthews_corrcoef(yte, p),
                     'Train AUC': roc_auc_score(ytr, score(m, Xtr)), 'CV AUC': f['cv_auc'], 'CV std': f['cv_std'],
                     'Time (s)': f['secs'], 'Best params': str(f['params'])})
    return pd.DataFrame(rows).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)

res_sp = evaluate(spatial, X.iloc[tr], y[tr], X.iloc[te], y[te])
res_sp.to_csv(f'{OUT}/table_spatial_results.csv', index=False)
best_name = res_sp.loc[0, 'Model']; best = spatial[best_name]['model']
print('Best model under spatial validation:', best_name)
show_table(res_sp.drop(columns='Best params'), 'spatial_results', 'Held-out block results (all models)', index=False)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.8))
for name in res_sp['Model']:
    s = score(spatial[name]['model'], X.iloc[te]); fpr, tpr, _ = roc_curve(y[te], s); pr, rc, _ = precision_recall_curve(y[te], s)
    lw = 2.4 if name == best_name else 1
    ax[0].plot(fpr, tpr, lw=lw, label=f'{name} ({roc_auc_score(y[te], s):.3f})'); ax[1].plot(rc, pr, lw=lw, label=f'{name} ({average_precision_score(y[te], s):.3f})')
ax[0].plot([0, 1], [0, 1], 'k--', lw=.8); ax[0].set_xlabel('False positive rate'); ax[0].set_ylabel('True positive rate')
ax[1].axhline(y[te].mean(), c='k', ls='--', lw=.8); ax[1].set_xlabel('Recall'); ax[1].set_ylabel('Precision')
for a in ax: a.legend(fontsize=6.5)
plt.tight_layout(); savefig('fig7_roc_pr')

fig, axs = plt.subplots(2, 5, figsize=(16, 6.4)); CM = {}
for a, name in zip(axs.ravel(), res_sp['Model']):
    cm = confusion_matrix(y[te], spatial[name]['model'].predict(X.iloc[te])); CM[name] = cm
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=a, xticklabels=['Stable', 'Landslide'], yticklabels=['Stable', 'Landslide'])
    a.set_xlabel(f'Predicted ({name})'); a.set_ylabel('Actual')
plt.tight_layout(); savefig('fig8_confusion_matrices')
RESULTS['confusion'] = {k: v.tolist() for k, v in CM.items()}

plt.figure(figsize=(9, 3.8)); idx = np.arange(len(res_sp))
plt.bar(idx - .2, res_sp['Train AUC'], .4, label='Training AUC', color='#dd8452'); plt.bar(idx + .2, res_sp['ROC-AUC'], .4, label='Spatial test AUC', color='#4c72b0')
for i, (a, b) in enumerate(zip(res_sp['Train AUC'], res_sp['ROC-AUC'])): plt.text(i, max(a, b) + .01, f'{a - b:+.2f}', ha='center', fontsize=7)
plt.xticks(idx, res_sp['Model'], rotation=40, ha='right'); plt.ylim(.4, 1.08); plt.legend(); savefig('fig9_train_vs_test')

### 7b. [NEW] Are the differences between models real? Block bootstrap and McNemar
The 3,505 test points sit in a limited number of blocks, so points are not independent. Confidence intervals are
obtained by resampling **whole test blocks** with replacement; all models are scored on the same resample so the
difference to the best model is paired.

In [ ]:
from scipy.stats import binomtest
yt = y[te]; tb = groups[te]; ub = np.unique(tb); idx_by_block = {b: np.where(tb == b)[0] for b in ub}
SC = {n: score(spatial[n]['model'], X.iloc[te]) for n in res_sp['Model']}
rb = np.random.default_rng(SEED); boot = []
for _ in range(N_BOOT):
    ii = np.concatenate([idx_by_block[b] for b in rb.choice(ub, len(ub), replace=True)])
    if yt[ii].min() == yt[ii].max(): continue
    boot.append({n: roc_auc_score(yt[ii], SC[n][ii]) for n in SC})
boot = pd.DataFrame(boot)
diff = boot.sub(boot[best_name], axis=0)
ci = pd.DataFrame({'AUC': res_sp.set_index('Model')['ROC-AUC'], 'lo': boot.quantile(.025), 'hi': boot.quantile(.975),
                   'dAUC_vs_best': diff.mean(), 'd_lo': diff.quantile(.025), 'd_hi': diff.quantile(.975),
                   'P(model >= best)': (diff >= 0).mean()}).loc[res_sp['Model']]

def mcnemar(a, b):
    pa, pb = spatial[a]['model'].predict(X.iloc[te]) == yt, spatial[b]['model'].predict(X.iloc[te]) == yt
    n01, n10 = int(np.sum(pa & ~pb)), int(np.sum(~pa & pb))
    return n01, n10, binomtest(min(n01, n10), n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
ci['McNemar p vs best'] = [np.nan if n == best_name else mcnemar(best_name, n)[2] for n in ci.index]
ci.to_csv(f'{OUT}/table_bootstrap_ci.csv'); RESULTS['bootstrap_ci'] = ci.round(4).reset_index().to_dict(orient='records')

key = lambda n: n.split(' (')[0].replace(' ', '').lower()
for n in ci.index:
    if 'Baseline' in n: continue
    pv(f'ci{key(n)}', f"{ci.loc[n, 'lo']:.3f}--{ci.loc[n, 'hi']:.3f}")
    if n != best_name:
        pv(f'd{key(n)}', f"{ci.loc[n, 'dAUC_vs_best']:+.3f} [{ci.loc[n, 'd_lo']:+.3f}, {ci.loc[n, 'd_hi']:+.3f}]")
        pv(f'mcn{key(n)}', f"{ci.loc[n, 'McNemar p vs best']:.3f}" if ci.loc[n, 'McNemar p vs best'] >= 0.001 else '<0.001')
        pv(f'pge{key(n)}', f"{100 * ci.loc[n, 'P(model >= best)']:.1f}")
pv('nboot', str(len(boot)))
show_table(ci, 'bootstrap_ci', 'Block-bootstrap AUC intervals and McNemar tests')

In [ ]:
# Visual [NEW]: AUC with 95 % block-bootstrap intervals (column format)
cc = ci.drop(index=[n for n in ci.index if 'Baseline' in n]).iloc[::-1]
fig, ax = plt.subplots(figsize=(4.2, 3.2))
ax.errorbar(cc['AUC'], range(len(cc)), xerr=[cc['AUC'] - cc['lo'], cc['hi'] - cc['AUC']], fmt='o', c='#4c72b0', capsize=3, ms=4)
ax.axvline(cc.loc[best_name, 'lo'], c='#c44e52', ls=':', lw=1)
ax.set_yticks(range(len(cc))); ax.set_yticklabels(cc.index, fontsize=7); ax.set_xlabel('Spatial test ROC-AUC (95 % block bootstrap)')
plt.tight_layout(); savefig('figA_bootstrap_ci_col')

### 7c. [NEW] Operating thresholds, calibration and the meaning of a score

In [ ]:
from sklearn.calibration import calibration_curve
sb = score(best, X.iloc[te]); fpr, tpr, thr = roc_curve(yt, sb)
def at_recall(r):
    k = int(np.argmax(tpr >= r)); p_ = sb >= thr[k]
    return dict(thr=thr[k], recall=tpr[k], fpr=fpr[k], precision=precision_score(yt, p_), false_alarms=int(np.sum(p_ & (yt == 0))),
                missed=int(np.sum(~p_ & (yt == 1))))
OP = {'default 0.5': dict(thr=0.5, recall=recall_score(yt, sb >= .5), fpr=np.mean(sb[yt == 0] >= .5), precision=precision_score(yt, sb >= .5),
                          false_alarms=int(np.sum((sb >= .5) & (yt == 0))), missed=int(np.sum((sb < .5) & (yt == 1)))),
      'recall 0.90': at_recall(0.90), 'recall 0.95': at_recall(0.95)}
j = int(np.argmax(tpr - fpr)); OP['Youden J'] = dict(thr=thr[j], recall=tpr[j], fpr=fpr[j], precision=precision_score(yt, sb >= thr[j]),
                                                     false_alarms=int(np.sum((sb >= thr[j]) & (yt == 0))), missed=int(np.sum((sb < thr[j]) & (yt == 1))))
OPdf = pd.DataFrame(OP).T; RESULTS['operating_points'] = OPdf.round(4).reset_index().to_dict(orient='records')
for nm, tag in [('recall 0.90', 'rninety'), ('recall 0.95', 'rninetyfive'), ('Youden J', 'youden')]:
    pv(f'{tag}thr', f"{OP[nm]['thr']:.2f}"); pv(f'{tag}prec', f"{OP[nm]['precision']:.3f}")
    pv(f'{tag}fa', f"{OP[nm]['false_alarms']:,}"); pv(f'{tag}fpr', f"{OP[nm]['fpr']:.3f}"); pv(f'{tag}rec', f"{OP[nm]['recall']:.3f}")

# Calibration: Brier score, and why a probability from a 1:1 sample is not an absolute probability
BR = {n: brier_score_loss(yt, spatial[n]['model'].predict_proba(X.iloc[te])[:, 1])
      for n in res_sp['Model'] if hasattr(spatial[n]['model'], 'predict_proba') and 'Baseline' not in n}
RESULTS['brier'] = {k: round(v, 4) for k, v in BR.items()}; pv('brierbest', f3(BR[best_name]))
prev = ls['Area_'].sum() / (area.area)          # share of the study area that is mapped landslide
pv('prevpct', f'{100 * prev:.3f}')
corr = lambda p: (p / (1 - p) * prev / (1 - prev)) / (1 + p / (1 - p) * prev / (1 - prev))
pv('priorseventy', f'{100 * corr(0.70):.2f}')
show_table(OPdf, 'operating_points', f'{best_name}: operating points on the test blocks'); print('Brier', {k: round(v, 3) for k, v in BR.items()}, '| area prevalence', PAPER['prevpct'], '%')

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
sns.histplot(x=sb, hue=np.where(yt == 1, 'Landslide', 'Stable'), bins=40, stat='density', common_norm=False, element='step', ax=ax[0], palette=['#4c72b0', '#c44e52'])
for name in list(BR)[:4]:
    pt, pp = calibration_curve(yt, spatial[name]['model'].predict_proba(X.iloc[te])[:, 1], n_bins=10)
    ax[1].plot(pp, pt, 'o-', ms=3, label=name)
ax[1].plot([0, 1], [0, 1], 'k--', lw=.8); ax[1].set_xlabel('Predicted probability (1:1 sample)'); ax[1].set_ylabel('Observed share'); ax[1].legend(fontsize=7)
plt.tight_layout(); savefig('fig9b_scores_calibration')

pr_, rc_, th_ = precision_recall_curve(yt, sb)
fig, ax = plt.subplots(figsize=(4.2, 2.8))
ax.plot(th_, pr_[:-1], c='#4c72b0', label='Precision'); ax.plot(th_, rc_[:-1], c='#c44e52', label='Recall')
for nm, mk in [('default 0.5', 'o'), ('recall 0.90', 's'), ('recall 0.95', '^')]:
    ax.axvline(OP[nm]['thr'], c='grey', ls=':', lw=.8); ax.plot(OP[nm]['thr'], OP[nm]['recall'], mk, c='k', ms=4, label=nm)
ax.set_xlabel(f'{best_name} score threshold'); ax.set_ylabel('Value'); ax.legend(fontsize=6); plt.tight_layout(); savefig('figI_thresholds_col')

In [ ]:
# [NEW] Does performance differ between the two terrain regimes found by K-means?
te_terr = X.iloc[te][TERRAIN]; reg = km.predict(scL.transform(te_terr))
REG = {}
for k in range(k_best):
    mk = reg == k
    if len(np.unique(yt[mk])) < 2: continue
    REG[k] = {'n': int(mk.sum()), 'landslide_share': float(yt[mk].mean()), 'mean_slope': float(te_terr['slope'][mk].mean()),
              'auc_best': roc_auc_score(yt[mk], sb[mk]), 'auc_lr': roc_auc_score(yt[mk], SC['Logistic Regression'][mk])}
REGdf = pd.DataFrame(REG).T; RESULTS['regime_auc'] = REGdf.round(4).reset_index().to_dict(orient='records')
order = REGdf.sort_values('mean_slope').index
for tag, k in zip(['mod', 'steep'], order):
    pv(f'reg{tag}auc', f3(REGdf.loc[k, 'auc_best'])); pv(f'reg{tag}lr', f3(REGdf.loc[k, 'auc_lr']))
    pv(f'reg{tag}share', f"{100 * REGdf.loc[k, 'landslide_share']:.1f}"); pv(f'reg{tag}n', f"{int(REGdf.loc[k, 'n']):,}")
show_table(REGdf, 'regime_auc', 'Test AUC within each K-means terrain regime')

In [ ]:
from sklearn.tree import plot_tree
fig, ax = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={'width_ratios': [1, 2.4]})
cs = coef.sort_values(); ax[0].barh(cs.index, cs.values, color=['#4c72b0' if v < 0 else '#c44e52' for v in cs.values]); ax[0].axvline(0, c='k', lw=.8)
small = DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, random_state=SEED).fit(X.iloc[tr], y[tr])
plot_tree(small, feature_names=FEATURES, class_names=['Stable', 'Landslide'], filled=True, rounded=True, fontsize=7, ax=ax[1], impurity=False, proportion=True)
plt.tight_layout(); savefig('fig9c_interpretable_models')

## 8. Random split vs spatial split

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(X, y, test_size=TEST_FRACTION, stratify=y, random_state=SEED)
print('Random cross-validation (StratifiedKFold):')
rand = fit_all(Xr_tr, yr_tr, None, StratifiedKFold(CV_FOLDS, shuffle=True, random_state=SEED))
res_rd = evaluate(rand, Xr_tr, yr_tr, Xr_te, yr_te); res_rd.to_csv(f'{OUT}/table_random_results.csv', index=False)
cmp = res_sp[['Model', 'ROC-AUC', 'F1']].merge(res_rd[['Model', 'ROC-AUC', 'F1']], on='Model', suffixes=(' spatial', ' random'))
cmp['AUC inflation'] = cmp['ROC-AUC random'] - cmp['ROC-AUC spatial']; cmp.to_csv(f'{OUT}/table_random_vs_spatial.csv', index=False)
show_table(cmp, 'random_vs_spatial', 'Held-out AUC and F1: random vs block split', index=False)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.4), gridspec_kw={'width_ratios': [1.6, 1, 1]}); idx = np.arange(len(cmp))
ax[0].bar(idx - .2, cmp['ROC-AUC random'], .4, label='Random split', color='#dd8452'); ax[0].bar(idx + .2, cmp['ROC-AUC spatial'], .4, label='Spatial blocks', color='#4c72b0')
ax[0].set_xticks(idx); ax[0].set_xticklabels(cmp['Model'], rotation=40, ha='right'); ax[0].set_ylim(.4, 1.02); ax[0].legend()
for _, r in cmp.iterrows(): ax[1].plot([0, 1], [r['ROC-AUC random'], r['ROC-AUC spatial']], 'o-', c='#c44e52' if r['Model'] == best_name else '#999999')
ax[1].set_xticks([0, 1]); ax[1].set_xticklabels(['Random', 'Spatial']); ax[1].set_ylabel('ROC-AUC')
ax[2].scatter(cmp['ROC-AUC spatial'], cmp['ROC-AUC random'], c='#4c72b0')
lo_ = min(cmp[['ROC-AUC spatial', 'ROC-AUC random']].min()) - .02; ax[2].plot([lo_, 1], [lo_, 1], 'k--', lw=.8)
ax[2].set_xlabel('Spatial AUC'); ax[2].set_ylabel('Random AUC'); plt.tight_layout(); savefig('fig10_random_vs_spatial')

### 8b. [NEW] Where does the random split inflate performance? CV stage, hyperparameters and stability

In [ ]:
CVC = pd.DataFrame({n: {'spatial CV': spatial[n]['cv_auc'], 'spatial CV std': spatial[n]['cv_std'],
                        'random CV': rand[n]['cv_auc'], 'random CV std': rand[n]['cv_std'],
                        'params spatial': str(spatial[n]['params']), 'params random': str(rand[n]['params'])}
                    for n in MODELS if 'Baseline' not in n}).T
CVC['CV inflation'] = CVC['random CV'].astype(float) - CVC['spatial CV'].astype(float)
CVC['std ratio'] = CVC['spatial CV std'].astype(float) / CVC['random CV std'].astype(float)
CVC = CVC.sort_values('CV inflation', ascending=False); CVC.to_csv(f'{OUT}/table_cv_random_vs_spatial.csv')
RESULTS['cv_random_vs_spatial'] = CVC.reset_index().to_dict(orient='records')
pv('cvinflmax', f"{CVC['CV inflation'].max():+.3f}"); pv('cvinflbest', f"{CVC.loc[best_name, 'CV inflation']:+.3f}")
pv('cvstdratio', f"{CVC['std ratio'].astype(float).median():.1f}")
show_table(CVC, 'cv_random_vs_spatial', 'Cross-validation stage: random vs block CV')

In [ ]:
# KNN: AUC against k under both CV schemes (distance weights) — the random scheme rewards small, local neighbourhoods
def curve(fitted, name, pname, fixed):
    t = fitted[name]['cv_table']; msk = np.ones(len(t), bool)
    for k_, v_ in fixed.items(): msk &= t[f'param_{k_}'].astype(str) == str(v_)
    t = t[msk]; return t[f'param_{pname}'].astype(float).values, t['mean_test_score'].values
fig, ax = plt.subplots(1, 2, figsize=(6.8, 2.8))
for fitted, lab, c in [(spatial, 'Spatial CV', '#4c72b0'), (rand, 'Random CV', '#dd8452')]:
    k_, a_ = curve(fitted, 'KNN', 'model__n_neighbors', {'model__weights': 'distance'}); ax[0].plot(k_, a_, 'o-', c=c, label=lab, ms=3)
ax[0].set_xlabel('k (distance-weighted)'); ax[0].set_ylabel('CV ROC-AUC'); ax[0].legend(fontsize=7)
nm_ = [n for n in CVC.index]; xx = np.arange(len(nm_))
ax[1].bar(xx - .2, CVC['spatial CV std'].astype(float), .4, color='#4c72b0', label='Spatial'); ax[1].bar(xx + .2, CVC['random CV std'].astype(float), .4, color='#dd8452', label='Random')
ax[1].set_xticks(xx); ax[1].set_xticklabels([n.split(' (')[0].replace('Gaussian ', '').replace('Logistic Regression', 'LR').replace('Gradient Boosting', 'GB').replace('Random Forest', 'RF').replace('Extra Trees', 'ET').replace('Decision Tree', 'DT') for n in nm_], rotation=45, fontsize=6)
ax[1].set_ylabel('CV fold std'); ax[1].legend(fontsize=7); plt.tight_layout(); savefig('figB_cv_random_vs_spatial_col')

### 8c. [NEW] Mechanism test: how close are test points to training points?
If random-split inflation comes from near-duplicate neighbours, a random test point should lie much closer to a
training point than a spatial-block test point does, and random-split AUC should be highest at short distances.

In [ ]:
P_ = data[['x', 'y']].values
nn_sp, _ = cKDTree(P_[tr]).query(P_[te]); Xr_idx_te = Xr_te.index.values; Xr_idx_tr = Xr_tr.index.values
nn_rd, _ = cKDTree(P_[Xr_idx_tr]).query(P_[Xr_idx_te])
pv('nnrandmed', f'{np.median(nn_rd):.0f}'); pv('nnspatmed', f'{np.median(nn_sp):.0f}')
pv('nnrandunder250', f'{100 * np.mean(nn_rd < 250):.1f}'); pv('nnspatunder250', f'{100 * np.mean(nn_sp < 250):.1f}')
# same-class nearest neighbour (a landslide's nearest training landslide)
def nn_same(tr_idx, te_idx):
    out = np.empty(len(te_idx))
    for k in (0, 1):
        a_, b_ = tr_idx[y[tr_idx] == k], te_idx[y[te_idx] == k]
        out[y[te_idx] == k] = cKDTree(P_[a_]).query(P_[b_])[0]
    return out
nns_rd, nns_sp = nn_same(Xr_idx_tr, Xr_idx_te), nn_same(tr, te)
pv('nnsamerandmed', f'{np.median(nns_rd):.0f}'); pv('nnsamespatmed', f'{np.median(nns_sp):.0f}')

bands = [0, 250, 500, 1000, 2000, 5000, np.inf]; s_rd = score(rand[best_name]['model'], Xr_te)
BAND = []
for lo_b, hi_b in zip(bands[:-1], bands[1:]):
    mk = (nn_rd >= lo_b) & (nn_rd < hi_b)
    if mk.sum() > 50 and len(np.unique(yr_te[mk])) == 2:
        BAND.append({'band_m': f'{lo_b:.0f}-{hi_b:.0f}', 'n': int(mk.sum()), 'auc_random_model': roc_auc_score(yr_te[mk], s_rd[mk])})
BAND = pd.DataFrame(BAND); RESULTS['auc_by_distance_band'] = BAND.round(4).to_dict(orient='records')
if len(BAND): pv('bandfirstauc', f3(BAND.iloc[0]['auc_random_model'])); pv('bandlastauc', f3(BAND.iloc[-1]['auc_random_model']))
show_table(BAND, 'auc_by_distance', 'Random-split AUC by distance to nearest training point', index=False)
fig, ax = plt.subplots(1, 2, figsize=(6.8, 2.8))
bins_ = np.logspace(1, 4.3, 40)
ax[0].hist(nn_rd, bins=bins_, alpha=.6, color='#dd8452', label='Random test'); ax[0].hist(nn_sp, bins=bins_, alpha=.6, color='#4c72b0', label='Spatial test')
ax[0].set_xscale('log'); ax[0].set_xlabel('Distance to nearest training point (m)'); ax[0].set_ylabel('Test points'); ax[0].legend(fontsize=7)
if len(BAND): ax[1].plot(range(len(BAND)), BAND['auc_random_model'], 'o-', c='#dd8452'); ax[1].set_xticks(range(len(BAND))); ax[1].set_xticklabels(BAND['band_m'], rotation=40, fontsize=6)
ax[1].axhline(res_sp.loc[0, 'ROC-AUC'], c='#4c72b0', ls='--', lw=1, label='Spatial test AUC'); ax[1].set_ylabel('Random-split test AUC'); ax[1].legend(fontsize=6)
plt.tight_layout(); savefig('figC_nn_distance_col')

In [ ]:
# [NEW] Repeated splits with fixed hyperparameters: is the +0.005 a stable effect or one draw?
REP_MODELS = {n: spatial[n]['model'] for n in [best_name, 'Random Forest', 'KNN', 'Logistic Regression'] if n in spatial}
rep = []
for r_ in range(N_REPEATS):
    a_tr, a_te = next(GroupShuffleSplit(1, test_size=TEST_FRACTION, random_state=100 + r_).split(X, y, groups))
    b_tr, b_te = train_test_split(np.arange(len(X)), test_size=TEST_FRACTION, stratify=y, random_state=100 + r_)
    for n, m_ in REP_MODELS.items():
        sa = score(clone(m_).fit(X.iloc[a_tr], y[a_tr]), X.iloc[a_te]); sb_ = score(clone(m_).fit(X.iloc[b_tr], y[b_tr]), X.iloc[b_te])
        rep.append({'rep': r_, 'Model': n, 'spatial': roc_auc_score(y[a_te], sa), 'random': roc_auc_score(y[b_te], sb_)})
rep = pd.DataFrame(rep); rep['inflation'] = rep['random'] - rep['spatial']
REP = rep.groupby('Model').agg(spatial_mean=('spatial', 'mean'), spatial_sd=('spatial', 'std'), random_mean=('random', 'mean'),
                               random_sd=('random', 'std'), infl_mean=('inflation', 'mean'), infl_sd=('inflation', 'std'),
                               infl_positive=('inflation', lambda v: int((v > 0).sum())))
RESULTS['repeated_splits'] = REP.round(4).reset_index().to_dict(orient='records'); REP.to_csv(f'{OUT}/table_repeated_splits.csv')
for n in REP.index:
    pv(f'rep{key(n)}', f"{REP.loc[n, 'infl_mean']:+.3f} $\\pm$ {REP.loc[n, 'infl_sd']:.3f}")
    pv(f'reppos{key(n)}', f"{REP.loc[n, 'infl_positive']}")
    pv(f'repspsd{key(n)}', f"{REP.loc[n, 'spatial_sd']:.3f}"); pv(f'reprdsd{key(n)}', f"{REP.loc[n, 'random_sd']:.3f}")
pv('nrepeats', str(N_REPEATS))
show_table(REP, 'repeated_splits', f'Repeated splits ({N_REPEATS}), fixed hyperparameters')

In [ ]:
# [NEW] Block-size sensitivity: CV AUC (5 folds, all data, fixed hyperparameters) as blocks grow
BS = []
cv_r = StratifiedKFold(CV_FOLDS, shuffle=True, random_state=SEED)
for n, m_ in REP_MODELS.items():
    a_ = [roc_auc_score(y[v], score(clone(m_).fit(X.iloc[t_], y[t_]), X.iloc[v])) for t_, v in cv_r.split(X, y)]
    BS.append({'Model': n, 'block_km': 0, 'auc': np.mean(a_), 'n_blocks': len(X)})
    for bk in BLOCK_SIZES_KM:
        gb_ = block_id(data['x'], data['y'], bk * 1000)
        if len(np.unique(gb_)) < CV_FOLDS: continue
        a_ = [roc_auc_score(y[v], score(clone(m_).fit(X.iloc[t_], y[t_]), X.iloc[v])) for t_, v in GroupKFold(CV_FOLDS).split(X, y, gb_)
              if len(np.unique(y[v])) == 2]
        BS.append({'Model': n, 'block_km': bk, 'auc': np.mean(a_), 'n_blocks': len(np.unique(gb_))})
BS = pd.DataFrame(BS); RESULTS['block_size'] = BS.round(4).to_dict(orient='records'); BS.to_csv(f'{OUT}/table_block_size.csv', index=False)
piv = BS.pivot(index='block_km', columns='Model', values='auc')
for n in piv.columns:
    pv(f'bs{key(n)}rand', f3(piv.loc[0, n])); pv(f'bs{key(n)}max', f3(piv[n].iloc[-1]))
    pv(f'bs{key(n)}drop', f"{piv.loc[0, n] - piv[n].iloc[-1]:.3f}")
pv('bslargest', f'{int(piv.index[-1])}')
fig, ax = plt.subplots(figsize=(4.2, 2.8))
for n in piv.columns: ax.plot(range(len(piv)), piv[n], 'o-', ms=3, label=n.split(' (')[0])
ax.set_xticks(range(len(piv))); ax.set_xticklabels(['random'] + [f'{int(b)}' for b in piv.index[1:]], fontsize=7)
ax.set_xlabel('CV block size (km)'); ax.set_ylabel('5-fold CV ROC-AUC'); ax.legend(fontsize=6); plt.tight_layout(); savefig('figD_block_size_col')
show_table(piv, 'block_size', 'CV AUC against block size (km; 0 = random CV)')

## 9. Feature importance, interactions and the aspect question

In [ ]:
from sklearn.inspection import permutation_importance
pi = permutation_importance(best, X.iloc[te], y[te], scoring='roc_auc', n_repeats=10, random_state=SEED, n_jobs=-1)
imp = pd.DataFrame({'feature': FEATURES, 'importance': pi.importances_mean, 'std': pi.importances_std}).sort_values('importance')
imp.to_csv(f'{OUT}/table_feature_importance.csv', index=False)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].barh(imp['feature'], imp['importance'], xerr=imp['std'], color='#55a868'); ax[0].set_xlabel('Drop in test ROC-AUC when shuffled')
rf = spatial['Random Forest']['model']; gini = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()
ax[1].barh(gini.index, gini.values, color='#8172b2'); plt.tight_layout(); savefig('fig11_feature_importance')

In [ ]:
# [NEW] Grouped permutation importance: permute collinear features together so shared information cannot leak back
GROUPS = {'Steepness (slope, TRI, relief)': ['slope', 'tri', 'relief'], 'Aspect (sin, cos)': ['aspect_sin', 'aspect_cos'],
          'Elevation': ['elevation'], 'Position (x, y)': ['x_km', 'y_km']}
base_auc = roc_auc_score(yt, sb); rg = np.random.default_rng(SEED); GI = []
for gname, cols in GROUPS.items():
    drops = []
    for _ in range(10):
        Xp = X.iloc[te].copy(); perm = rg.permutation(len(Xp)); Xp[cols] = Xp[cols].values[perm]
        drops.append(base_auc - roc_auc_score(yt, score(best, Xp)))
    GI.append({'group': gname, 'drop': np.mean(drops), 'sd': np.std(drops)})
GI = pd.DataFrame(GI).sort_values('drop'); RESULTS['grouped_importance'] = GI.round(4).to_dict(orient='records')
for _, r in GI.iterrows(): pv('gi' + r['group'].split(' ')[0].lower(), f"{r['drop']:.3f}")
# individual-feature numbers for the paper
for f_ in FEATURES: pv('pi' + f_.replace('_', ''), f"{imp.set_index('feature').loc[f_, 'importance']:.3f}")
fig, ax = plt.subplots(figsize=(4.2, 2.4))
ax.barh(GI['group'], GI['drop'], xerr=GI['sd'], color='#55a868'); ax.set_xlabel('Drop in test AUC (group permuted)')
plt.tight_layout(); savefig('figE_grouped_importance_col')
show_table(GI, 'grouped_importance', 'Grouped permutation importance', index=False)

In [ ]:
from sklearn.inspection import PartialDependenceDisplay
top_feats = [f for f in imp.sort_values('importance', ascending=False)['feature'] if f in TERRAIN][:4]
fig, ax = plt.subplots(1, len(top_feats), figsize=(4 * len(top_feats), 3.4))
PartialDependenceDisplay.from_estimator(best, X.iloc[tr].sample(min(3000, len(tr)), random_state=SEED), top_feats, kind='both', subsample=200,
                                        ax=ax, random_state=SEED, ice_lines_kw={'alpha': .08, 'color': '#999999'}, pd_line_kw={'color': '#c44e52', 'lw': 2.5})
plt.tight_layout(); savefig('fig11b_partial_dependence')
# [NEW] two-way interaction: ruggedness x north-south aspect
fig, ax = plt.subplots(figsize=(4.2, 3.4))
PartialDependenceDisplay.from_estimator(best, X.iloc[tr].sample(min(3000, len(tr)), random_state=SEED), [('tri', 'aspect_cos')], ax=ax,
                                        random_state=SEED, contour_kw={'cmap': 'RdYlGn_r'})
plt.tight_layout(); savefig('figJ_pdp_interaction_col')

In [ ]:
# Ablations (same split). Terrain-only keeps the original tuned procedure; the others refit the best configuration.
abl_est, abl_grid = MODELS[best_name]
gs = GridSearchCV(abl_est, abl_grid or {}, cv=GroupKFold(CV_FOLDS), scoring='roc_auc', n_jobs=-1)
gs.fit(X.iloc[tr][TERRAIN], y[tr], groups=groups[tr]); auc_terrain = roc_auc_score(y[te], score(gs.best_estimator_, X.iloc[te][TERRAIN]))
ABL = {'All 8 features': res_sp.loc[0, 'ROC-AUC'], 'Terrain only (no x, y)': auc_terrain}
for lab, drop in [('Without aspect', ['aspect_sin', 'aspect_cos']), ('Without steepness', ['slope', 'tri', 'relief']),
                  ('Without elevation', ['elevation']), ('Steepness only', [f for f in FEATURES if f not in ('slope', 'tri', 'relief')])]:
    cols = [f for f in FEATURES if f not in drop]
    ABL[lab] = roc_auc_score(y[te], score(clone(best).fit(X.iloc[tr][cols], y[tr]), X.iloc[te][cols]))
ABL = pd.Series(ABL); RESULTS['ablation'] = ABL.round(4).to_dict()
for k_, tag in [('Terrain only (no x, y)', 'ablterrain'), ('Without aspect', 'ablnoaspect'), ('Without steepness', 'ablnosteep'),
                ('Without elevation', 'ablnoelev'), ('Steepness only', 'ablsteeponly')]: pv(tag, f3(ABL[k_]))
plt.figure(figsize=(5, 3)); plt.barh(ABL.index[::-1], ABL.values[::-1], color='#4c72b0'); plt.xlim(.5, .9); plt.xlabel('Spatial test ROC-AUC')
savefig('fig11c_ablation'); show_table(ABL.to_frame('ROC-AUC'), 'ablation', 'Ablation (spatial test AUC)')

### 9b. [NEW] Is the aspect effect physical, or a winter-image visibility artefact?
Aspect and illumination are almost the same variable in one scene, so a single-date inventory cannot separate them
completely. Two partial tests are possible:
1. **Illumination gradient:** landslide-to-background ratio across illumination (cos *i*) bins.
2. **Cast shadow with aspect held fixed:** within each aspect octant, compare the share of landslides and of
   background points lying in a ridge's cast shadow. Physical controls do not change at the boundary of a shadow cast at
   one instant, but detectability does. A Mantel–Haenszel odds ratio < 1 means landslides are *under-mapped* in shadow.

In [ ]:
data['oct'] = (((data['aspect'] + 22.5) % 360) // 45).astype(int)
OCT = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
tab = []; num = den = 0.0
for k in range(8):
    s_ = data[data.oct == k]
    a = int(((s_.label == 1) & s_.shadow).sum()); b = int(((s_.label == 1) & ~s_.shadow).sum())
    c = int(((s_.label == 0) & s_.shadow).sum()); d_ = int(((s_.label == 0) & ~s_.shadow).sum()); n_ = a + b + c + d_
    if n_: num += a * d_ / n_; den += b * c / n_
    tab.append({'aspect': OCT[k], 'LS in shadow %': 100 * a / max(a + b, 1), 'BG in shadow %': 100 * c / max(c + d_, 1),
                'LS n': a + b, 'BG n': c + d_})
tab = pd.DataFrame(tab); MH = num / den if den else np.nan
RESULTS['shadow_by_aspect'] = tab.round(2).to_dict(orient='records'); RESULTS['mantel_haenszel_or'] = round(float(MH), 3)
pv('mhor', f'{MH:.2f}')
pv('lsshadow', f"{100 * data.loc[data.label == 1, 'shadow'].mean():.1f}"); pv('bgshadow', f"{100 * data.loc[data.label == 0, 'shadow'].mean():.1f}")
pv('lscosimed', f"{data.loc[data.label == 1, 'cos_i'].median():.2f}"); pv('bgcosimed', f"{data.loc[data.label == 0, 'cos_i'].median():.2f}")
pv('lsnorthpct', f"{100 * (data.loc[data.label == 1, 'oct'] == 0).mean():.1f}"); pv('bgnorthpct', f"{100 * (data.loc[data.label == 0, 'oct'] == 0).mean():.1f}")
# bootstrap CI for the MH odds ratio (resample points within class)
rmh = np.random.default_rng(SEED); mh_b = []
for _ in range(300 if not QUICK else 50):
    s_ = data.sample(frac=1, replace=True, random_state=int(rmh.integers(1e9))); nu = de = 0.0
    for k in range(8):
        t_ = s_[s_.oct == k]; a = ((t_.label == 1) & t_.shadow).sum(); b = ((t_.label == 1) & ~t_.shadow).sum()
        c = ((t_.label == 0) & t_.shadow).sum(); d_ = ((t_.label == 0) & ~t_.shadow).sum(); n_ = a + b + c + d_
        if n_: nu += a * d_ / n_; de += b * c / n_
    if de: mh_b.append(nu / de)
pv('mhorci', f'{np.quantile(mh_b, .025):.2f}--{np.quantile(mh_b, .975):.2f}')

data['cosi_bin'] = pd.qcut(data['cos_i'], 10, duplicates='drop')
ratio = data.groupby('cosi_bin', observed=True)['label'].mean()
fig, ax = plt.subplots(1, 2, figsize=(6.8, 2.8))
ax[0].plot([iv.mid for iv in ratio.index], ratio.values, 'o-', c='#c44e52'); ax[0].axhline(.5, c='k', ls=':', lw=.8)
ax[0].set_xlabel('Illumination cos i at overpass'); ax[0].set_ylabel('Landslide share in bin')
xx = np.arange(8); ax[1].bar(xx - .2, tab['LS in shadow %'], .4, color='#c44e52', label='Landslides'); ax[1].bar(xx + .2, tab['BG in shadow %'], .4, color='#4c72b0', label='Background')
ax[1].set_xticks(xx); ax[1].set_xticklabels(OCT, fontsize=7); ax[1].set_ylabel('% in cast shadow'); ax[1].legend(fontsize=6)
plt.tight_layout(); savefig('figF_illumination_col')
print('Mantel-Haenszel OR (landslide | shadow, aspect-stratified):', PAPER['mhor'], PAPER['mhorci'])
show_table(tab, 'shadow_by_aspect', 'Cast shadow by aspect octant', index=False, fmt='{:.1f}')

## 10. Basin-wide relative susceptibility map
The best model is refitted on all samples and applied to a 1 km grid; cells are split into tertiles.
**[NEW]** The original density check is in-sample (the map model has seen every landslide it is scored against).
A spatially **out-of-fold** map is therefore built: each grid cell is predicted by a model trained without its block.

In [ ]:
final = MODELS[best_name][0].set_params(**spatial[best_name]['params']).fit(X, y)
gx, gy = np.meshgrid(np.arange(minx, maxx, GRID_SPACING_M), np.arange(miny, maxy, GRID_SPACING_M))
grid = pd.DataFrame({'x': gx.ravel(), 'y': gy.ravel()}); grid = grid[shapely.contains_xy(area, grid.x.values, grid.y.values)]
glon, glat = to_ll.transform(grid.x.values, grid.y.values); gt = terrain(glon, glat); gt.index = grid.index
grid = pd.concat([grid, gt], axis=1).dropna(); grid = grid[grid.elevation <= MAX_ELEV_M].copy()
grid['aspect_sin'], grid['aspect_cos'] = np.sin(np.radians(grid.aspect)), np.cos(np.radians(grid.aspect))
grid['x_km'], grid['y_km'] = grid.x / 1000, grid.y / 1000
grid['p'] = score(final, grid[FEATURES]); q1, q2 = grid['p'].quantile([1/3, 2/3])
grid['class'] = pd.cut(grid['p'], [-np.inf, q1, q2, np.inf], labels=['Low', 'Moderate', 'High'])
pv('qone', f'{q1:.3f}'); pv('qtwo', f'{q2:.3f}'); pv('gridcells', f'{len(grid):,}')
_, nn = cKDTree(grid[['x', 'y']].values).query(ls_feat[['x', 'y']].values); cell_area = (GRID_SPACING_M / 1000) ** 2

def density(cls):
    dd = grid.groupby(cls, observed=False).size().rename('cells').to_frame(); dd['area_km2'] = dd['cells'] * cell_area
    dd['landslides'] = grid[cls].iloc[nn].value_counts().reindex(dd.index).fillna(0).astype(int)
    dd['share_of_landslides_%'] = (100 * dd['landslides'] / dd['landslides'].sum()).round(1)
    dd['landslides_per_km2'] = (dd['landslides'] / dd['area_km2']).round(3); return dd
dens = density('class'); dens.to_csv(f'{OUT}/table_density_per_class.csv'); grid.to_csv(f'{OUT}/risk_grid.csv', index=False)
show_table(dens, 'density_in_sample', 'Landslides per tertile (in-sample map)')

In [ ]:
# [NEW] Spatially out-of-fold map
grid['block'] = block_id(grid.x, grid.y); allb = np.unique(np.r_[data.block.unique(), grid.block.unique()])
fold_of = dict(zip(allb, np.random.default_rng(SEED).permutation(len(allb)) % CV_FOLDS))
data['mfold'] = data.block.map(fold_of); grid['mfold'] = grid.block.map(fold_of); grid['p_oof'] = np.nan
for f_ in range(CV_FOLDS):
    m_ = clone(final).fit(X[data.mfold != f_], y[data.mfold != f_]); sel = grid.mfold == f_
    if sel.any(): grid.loc[sel, 'p_oof'] = score(m_, grid.loc[sel, FEATURES])
o1, o2 = grid['p_oof'].quantile([1/3, 2/3]); grid['class_oof'] = pd.cut(grid['p_oof'], [-np.inf, o1, o2, np.inf], labels=['Low', 'Moderate', 'High'])
dens_oof = density('class_oof'); dens_oof.to_csv(f'{OUT}/table_density_per_class_oof.csv')
pv('oofhighshare', f"{dens_oof.loc['High', 'share_of_landslides_%']:.1f}"); pv('ooflowshare', f"{dens_oof.loc['Low', 'share_of_landslides_%']:.1f}")
pv('oofratio', f"{dens_oof.loc['High', 'landslides_per_km2'] / dens_oof.loc['Low', 'landslides_per_km2']:.1f}")
pv('oofhighdens', f"{dens_oof.loc['High', 'landslides_per_km2']:.3f}"); pv('ooflowdens', f"{dens_oof.loc['Low', 'landslides_per_km2']:.3f}")
pv('mapagree', f"{100 * np.mean(grid['class'].astype(str) == grid['class_oof'].astype(str)):.1f}")
pv('mapspearman', f"{grid['p'].corr(grid['p_oof'], method='spearman'):.3f}")

def rate_curve(p):
    order = np.argsort(-grid[p].values); rank = np.empty(len(order)); rank[order] = np.arange(len(order))
    cum_area = np.arange(1, len(order) + 1) / len(order)
    hits = np.bincount(rank[nn].astype(int), minlength=len(order)); cum_ls = np.cumsum(hits) / hits.sum()
    return cum_area, cum_ls, np.trapezoid(cum_ls, cum_area) if hasattr(np, 'trapezoid') else np.trapz(cum_ls, cum_area)
ca1, cl1, sr_auc = rate_curve('p'); ca2, cl2, pr_auc = rate_curve('p_oof')
pv('successauc', f3(sr_auc)); pv('predauc', f3(pr_auc))
pv('top10success', f"{100 * cl1[int(0.1 * len(cl1)) - 1]:.1f}"); pv('top10pred', f"{100 * cl2[int(0.1 * len(cl2)) - 1]:.1f}")
RESULTS['map_validation'] = {'success_rate_auc': round(sr_auc, 4), 'prediction_rate_auc': round(pr_auc, 4),
                             'density_oof': dens_oof.reset_index().to_dict(orient='records')}
fig, ax = plt.subplots(1, 2, figsize=(6.8, 2.9))
ax[0].plot(ca1 * 100, cl1 * 100, c='#dd8452', label=f'Success (in-sample) {sr_auc:.3f}'); ax[0].plot(ca2 * 100, cl2 * 100, c='#4c72b0', label=f'Prediction (out-of-fold) {pr_auc:.3f}')
ax[0].plot([0, 100], [0, 100], 'k:', lw=.8); ax[0].set_xlabel('% of area (highest score first)'); ax[0].set_ylabel('% of landslides'); ax[0].legend(fontsize=6)
xx = np.arange(3); ax[1].bar(xx - .2, dens['landslides_per_km2'], .4, color='#dd8452', label='In-sample'); ax[1].bar(xx + .2, dens_oof['landslides_per_km2'], .4, color='#4c72b0', label='Out-of-fold')
ax[1].set_xticks(xx); ax[1].set_xticklabels(dens.index); ax[1].set_ylabel('Landslides per km²'); ax[1].legend(fontsize=6)
plt.tight_layout(); savefig('figG_rate_curves_col'); show_table(dens_oof, 'density_oof', 'Landslides per tertile (out-of-fold map)')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5.2))
sc_ = ax[0].scatter(grid.x / 1000, grid.y / 1000, c=grid.p, s=1.4, marker='s', cmap='RdYlGn_r'); fig.colorbar(sc_, ax=ax[0], shrink=.8, label='Predicted score')
cmap = {'Low': '#1a9850', 'Moderate': '#fee08b', 'High': '#d73027'}
for k, v in cmap.items():
    s = grid[grid['class'] == k]; ax[1].scatter(s.x / 1000, s.y / 1000, s=1.4, c=v, marker='s', label=k)
ax[1].scatter(ls_feat.x / 1000, ls_feat.y / 1000, s=.3, c='k', alpha=.3, label='Mapped landslides'); ax[1].legend(markerscale=6, loc='lower left')
for a in ax: a.set_aspect('equal'); a.set_xlabel('Easting (km)'); a.set_ylabel('Northing (km)')
plt.tight_layout(); savefig('fig12_risk_map')
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8)); cols = [cmap[c] for c in dens.index]
ax[0].hist(grid.p, bins=50, color='#999999'); [ax[0].axvline(v, c='k', ls='--') for v in (q1, q2)]; ax[0].set_xlabel('Score')
ax[1].bar(dens.index, dens['landslides_per_km2'], color=cols); ax[1].set_ylabel('Landslides per km²')
ax[2].bar(dens.index, dens['share_of_landslides_%'], color=cols); ax[2].plot(dens.index, [100 / 3] * 3, 'k--'); ax[2].set_ylabel('%')
plt.tight_layout(); savefig('fig12b_density_per_class')

## 11. Prototype: predict relative susceptibility for any location

In [ ]:
def predict_location(lat, lon, show=True):
    t = terrain([lon], [lat])
    if t.isna().any(axis=None): return 'Location is outside the DEM extent.'
    x, y_ = to_utm.transform(lon, lat)
    if not area.contains(Point(x, y_)) or t.elevation[0] > MAX_ELEV_M:
        return "Location is outside the study area or above 4,500 m (outside the model's training range)."
    row = t.assign(aspect_sin=np.sin(np.radians(t.aspect)), aspect_cos=np.cos(np.radians(t.aspect)), x_km=x / 1000, y_km=y_ / 1000)
    p = float(score(final, row[FEATURES])[0]); cls = 'High' if p > q2 else 'Moderate' if p > q1 else 'Low'
    pct = float((grid['p'] < p).mean() * 100)
    if show:
        fig, ax = plt.subplots(1, 2, figsize=(12, 3.6), gridspec_kw={'width_ratios': [2, 1]})
        ax[0].scatter(grid.x / 1000, grid.y / 1000, c=grid['class'].map(cmap), s=1, marker='s')
        ax[0].scatter(x / 1000, y_ / 1000, s=250, marker='*', c='blue', edgecolor='w', zorder=5); ax[0].set_aspect('equal')
        ax[0].set_title(f'({lat:.3f}, {lon:.3f}) → {cls}'); ax[1].hist(grid.p, bins=50, color='#cccccc'); ax[1].axvline(p, c='blue', lw=2)
        [ax[1].axvline(v, c='k', ls='--') for v in (q1, q2)]; ax[1].set_title(f'Score {p:.3f} (percentile {pct:.0f})'); plt.tight_layout(); plt.show()
    return {'score': round(p, 3), 'basin_percentile': round(pct, 1), 'relative_class': cls, **row[TERRAIN].round(1).iloc[0].to_dict()}
ex = predict_location(27.6, 86.0); print(ex)
if isinstance(ex, dict): pv('protoscore', f"{ex['score']:.3f}"); pv('protopct', f"{ex['basin_percentile']:.0f}"); pv('protoclass', ex['relative_class'])

In [ ]:
try:
    import ipywidgets as w
    from IPython.display import display
    la, lo, out = w.FloatText(27.6, description='Latitude'), w.FloatText(86.0, description='Longitude'), w.Output()
    btn = w.Button(description='Predict', button_style='danger')
    def _go(_):
        out.clear_output()
        with out: print(predict_location(la.value, lo.value))
    btn.on_click(_go); display(w.VBox([la, lo, btn, out]))
except Exception as e:
    print('ipywidgets not available:', e)

## 12. Save results and write `paper_values.tex`
All images are saved in `figures/`. Copy that folder and `outputs/paper_values.tex` next to the paper and recompile.

In [ ]:
r0 = res_sp.loc[0]; row_r = cmp.set_index('Model').loc[best_name]
RESULTS.update({'best_model': best_name, 'best_params': spatial[best_name]['params'],
  'spatial_test': {k: round(float(r0[k]), 3) for k in ['Accuracy', 'Balanced acc.', 'Precision', 'Recall', 'F1', 'ROC-AUC', 'PR-AUC', 'MCC', 'Train AUC', 'CV AUC', 'CV std']},
  'random_split_auc_best': round(float(row_r['ROC-AUC random']), 3), 'auc_inflation_best': round(float(row_r['AUC inflation']), 3),
  'mean_auc_inflation_all_models': round(float(cmp.loc[cmp.Model != 'Baseline (majority)', 'AUC inflation'].mean()), 3),
  'feature_importance': imp.set_index('feature')['importance'].round(4).to_dict(), 'auc_terrain_only': round(float(auc_terrain), 3),
  'density_per_class': dens.reset_index().to_dict(orient='records'),
  'all_models_spatial': res_sp.drop(columns='Best params').round(3).to_dict(orient='records')})
with open(f'{OUT}/results.json', 'w') as fh: json.dump(RESULTS, fh, indent=2, default=str)

pv('demsource', RESULTS['dem_source'].split(' (')[0])
def tex_escape(s): return str(s).replace('%', r'\%').replace('_', r'\_').replace('&', r'\&')
with open(f'{OUT}/paper_values.tex', 'w') as fh:
    fh.write('% Auto-generated by koshi_landslide.ipynb, Section 12. Do not edit by hand.\n')
    for k_, v_ in sorted(PAPER.items()):
        v_ = v_ if '$' in str(v_) else tex_escape(v_)
        fh.write(f'\\expandafter\\def\\csname pv@{k_}\\endcsname{{{v_}}}\n')
print(f'{len(PAPER)} values written to {OUT}/paper_values.tex')
print(pd.Series(PAPER).head(20).to_string(), '\n...')

### 12b. Column versions of the wide figures
Wide multi-panel figures are cut at the white space between their panels and re-arranged into a grid that fits
one column of the two-column paper. Output: `figures/<name>_col.png`.

In [ ]:
import numpy as np
from PIL import Image

def _gaps(mask_profile, min_run=8):
    runs=[]; start=None
    for i,v in enumerate(mask_profile):
        if v and start is None: start=i
        if not v and start is not None:
            runs.append((start,i)); start=None
    if start is not None: runs.append((start,len(mask_profile)))
    return [r for r in runs if r[1]-r[0]>=min_run]

def split(img, n, axis):
    '''Split img into n panels along axis (1 = side by side, 0 = stacked) at the widest white gaps.'''
    a=np.asarray(img.convert('L'))
    white=(a>245)
    prof=white.all(axis=0) if axis==1 else white.all(axis=1)
    L=len(prof)
    runs=[r for r in _gaps(prof) if r[0]>0 and r[1]<L]       # interior gaps only
    runs=sorted(sorted(runs,key=lambda r:r[1]-r[0],reverse=True)[:n-1])
    cuts=[0]+[ (r[0]+r[1])//2 for r in runs ]+[L]
    out=[]
    for c0,c1 in zip(cuts[:-1],cuts[1:]):
        box=(c0,0,c1,img.height) if axis==1 else (0,c0,img.width,c1)
        out.append(trim(img.crop(box)))
    return out

def split_at(img, fracs, axis):
    '''Cut at the white gap nearest to each fraction of the width (axis 1) or height (axis 0).'''
    a=np.asarray(img.convert('L')); white=(a>245)
    prof=white.all(axis=0) if axis==1 else white.all(axis=1); L=len(prof)
    runs=[r for r in _gaps(prof,min_run=4) if r[0]>0 and r[1]<L]
    cuts=[0]
    for f in fracs:
        t=f*L; best=min(runs,key=lambda r:abs((r[0]+r[1])/2-t)); cuts.append((best[0]+best[1])//2)
    cuts.append(L); out=[]
    for c0,c1 in zip(cuts[:-1],cuts[1:]):
        box=(c0,0,c1,img.height) if axis==1 else (0,c0,img.width,c1)
        out.append(trim(img.crop(box)))
    return out

def trim(im, pad=12):
    a=np.asarray(im.convert('L')); nz=np.where(a<245)
    if len(nz[0])==0: return im
    y0,y1,x0,x1=nz[0].min(),nz[0].max(),nz[1].min(),nz[1].max()
    return im.crop((max(x0-pad,0),max(y0-pad,0),min(x1+pad,im.width),min(y1+pad,im.height)))

def grid(panels, ncol, gap=20):
    rows=[panels[i:i+ncol] for i in range(0,len(panels),ncol)]
    cw=max(p.width for p in panels)
    W=ncol*cw+(ncol-1)*gap
    rh=[max(p.height for p in r) for r in rows]
    H=sum(rh)+gap*(len(rows)-1)
    canvas=Image.new('RGB',(W,H),'white'); y=0
    for r,h in zip(rows,rh):
        rw=len(r)*cw+(len(r)-1)*gap; x=(W-rw)//2
        for p in r:
            canvas.paste(p,(x+(cw-p.width)//2,y+(h-p.height)//2)); x+=cw+gap
        y+=h+gap
    return canvas

# name: (column fractions, row fractions, output columns)
SPECS={
 'fig7_roc_pr':               ([1/2],[],1),
 'fig11_feature_importance':  ([1/2],[],1),
 'fig12_risk_map':            ([1/2],[],1),
 'fig9c_interpretable_models':([1/3.4],[],1),
 'fig5_pca':                  ([1/3,2/3],[],2),
 'fig5b_kmeans':              ([1/3,2/3],[],2),
 'fig11b_partial_dependence': ([1/4,2/4,3/4],[],2),
 'fig12b_density_per_class':  ([1/3,2/3],[],2),
 'fig3_class_distributions':  ([1/3,2/3],[1/2],2),
 'fig8_confusion_matrices':   ([1/5,2/5,3/5,4/5],[1/2],3),
}

def make(src_dir, out_dir):
    import os
    os.makedirs(out_dir, exist_ok=True)
    for name,(cf,rf,ncol) in SPECS.items():
        p=f'{src_dir}/{name}.png'
        if not os.path.exists(p): continue
        im=Image.open(p).convert('RGB')
        rows=split_at(im,rf,0) if rf else [im]
        panels=[]
        for row in rows: panels+=split_at(row,cf,1)
        grid(panels,ncol).save(f'{out_dir}/{name}_col.png')
        print(name,[pp.size for pp in panels])

make(FIG, FIG)

In [ ]:
import shutil
shutil.copy(f'{OUT}/paper_values.tex', f'{FIG}/../paper_values.tex') if os.path.exists(f'{OUT}/paper_values.tex') else None
for d_ in (OUT, FIG): shutil.make_archive(f'koshi_{d_}', 'zip', d_)
try:
    from google.colab import files; files.download('koshi_outputs.zip'); files.download('koshi_figures.zip')
except ImportError:
    print('Saved koshi_outputs.zip and koshi_figures.zip')